# Imports

Please run these imports before going to the evaluation you want

In [ ]:
import importlib
from google3.third_party.tensorboard import notebook
%load_ext google3.learning.brain.tensorboard.notebook.extension
import jax
# import pygrain
from colabtools import drive
from scenic.dataset_lib import dataset_utils
import tensorflow_datasets as tfds
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))
import matplotlib.pyplot as plt
from etils import ecolab
import numpy as np
import tensorflow.experimental.numpy as tnp
from typing import List, Iterable, Dict, Any, Tuple
from google3.geo.imagery.streetview.store.shared.python import ids as gsv_ids
from google3.perftools.accelerators.xprof.api.colab import xprof

import mediapy

from scenic.train_lib import train_utils
from flax import jax_utils
ecolab.auto_plot_array()  # Display np.array as images/video
tnp.experimental_enable_numpy_behavior()  # Activate TF numpy behavior

jax.device_count(), jax.local_device_count()

In [ ]:
from clu import platform
import jax
import jax.numpy as jnp

from unigeoclip.scenic import load
from unigeoclip.scenic import trainer
from unigeoclip.scenic import datasets

import unigeoclip
from scaling_geoloc.utils import misc


def write_note(
    note: str,
):
  """Writes a note to the workdir."""
  if jax.process_index() == 0:
    platform.work_unit().set_notes(note)


In [ ]:
BASEPATH = "unigeoclip.scenic.configs.{}"

def get_config(config_name = "eval_geolocation_retrieval"):
  config = importlib.import_module(BASEPATH.format(config_name)).get_config().unlock() #bedenl
  # config = importlib.import_module(BASEPATH.format(config_name)).get_config(location='amsterdam').unlock()
  config.device_count = jax.device_count()
  config = config.copy_and_resolve_references()
  return config

# Qualitative plots

## Spatial PCA of latlng encoder

These cells given a xid in and a list of tokens return the PCA of the embedding of the center of s2cells at level 16 computed from the list of tokens.

In [ ]:
experiment_config = (211109853, 1)

_, model, train_state = trainer.load_from_xm(*experiment_config)
train_state = model.replicate_state(train_state)

In [ ]:
# New York
tokens = ['89c245c','89c2454','89c244c','89c2434','89c2f44','89c2f4c','89c25d','89c2594','89c258c','89c25bc','89c25b4','89c25ac',
          '89c24ff','89c24f9','89c25a5','89c25a3','89c2599','89c259b','89c2585','89c259c4','89c259cc','89c259ec','89c259f4','89c25a1c',
          '89c25a14','89c25a0d','89c25a0f','89c2f67','89c2f61','89c2f69','89c25864','89c2f624','89c2f63c','89c2f644','89c2f62c',
          '89c2f5d','89c2f5e4','89c2f5fd','89c2f39','89c2f3f','89c2f37','89c2f35','89c25e4','89c25ec','89c25fc','89c25f4','89c2604',
          '89c261c','89c260c','89c2614','89c2675','89c2673','89c266d','89c266b','89c2443','89c2445','89c2447','89c28ab','89c28a9',
          '89c28a7','89c2f5b','89c2f51',]
# Los Angeles
tokens = ["80c2b8543","80c2b8554","80c2b855c","80c2b8564","80c2b8569","80c2b856f","80c2b8574","80c2b857c","80c2b859","80c2b85bd",
          "80c2b85bf","80c2b85c1","80c2b85c3","80c2b85c5","80c2b85dc","80c2b85f","80c2b861","80c2b863","80c2b865","80c2b8665",
          "80c2b8667","80c2b866c","80c2b8674","80c2b867b","80c2b88c","80c2b891","80c2b893","80c2b895","80c2b8964","80c2b896c",
          "80c2b8974","80c2b8979","80c2b897b","80c2b897d","80c2b8b05","80c2b8b07","80c2b8b0c","80c2b8b14","80c2b8b19","80c2b8b1b",
          "80c2b8b23","80c2b8b25","80c2b8b3c","80c2b8b44","80c2b8b49","80c2b8b4f","80c2b8b51","80c2b8b55","80c2b8b57","80c2b8b5c",
          "80c2b8b7","80c2b8b9","80c2b8ba4","80c2b8ba9","80c2b8baf","80c2b8bb1","80c2b8bb7","80c2b8bbc","80c2b8bc4","80c2b8bc9",
          "80c2b8bcf","80c2b8bd1","80c2b8bd7","80c2b8bdc","80c2b8bf","80c2b8d","80c2b8f","80c2b904","80c2b9085","80c2b9087",
          "80c2b908c","80c2b9094","80c2b909b","80c2b90b1","80c2b90b5","80c2b90b7","80c2b90bc","80c2b90d","80c2b90f","80c2b914",
          "80c2b91c","80c2b93","80c2b95","80c2b964","80c2b96c","80c2b9704","80c2b9709","80c2b970f","80c2b9711","80c2b9715","80c2b9717",
          "80c2b971c","80c2b973","80c2b9749","80c2b974b","80c2b97a5","80c2b97a7","80c2b97ac","80c2b97b4","80c2b97bc","80c2b97d",
          "80c2b97e1","80c2b97e7","80c2b97e9","80c2b9aa1","80c2b9aa7","80c2b9aa9","80c2b9aab","80c2b9aad","80c2bb8d3","80c2bb8d5",
          "80c2bb8d7","80c2bb8d9","80c2bb905","80c2bb907","80c2bb90c","80c2bb914","80c2bb91c","80c2bb93","80c2bb95","80c2bb97",
          "80c2bb99","80c2bb9a4","80c2bb9bd","80c2bb9bf","80c2bb9ec","80c2bb9f3","80c2bb9f5","80c2bba24","80c2bba2c","80c2bba34",
          "80c2bba39","80c2bba3b","80c2bba4c","80c2bba53","80c2bbb77","80c2bbb79","80c2bbb7f","80c2bbb9","80c2bbba4","80c2bbbad",
          "80c2bbbaf","80c2bbbb4","80c2bbbbc","80c2bbbd","80c2bbbf","80c2bbc4","80c2bbc84","80c2bbc8c","80c2bbc93","80c2bbce4",
          "80c2bbced","80c2bbcef","80c2bbcf4","80c2bbcfc","80c2bbd1","80c2bbd21","80c2bbd23","80c2bbd39","80c2bbd3d","80c2bbd3f",
          "80c2bbd44","80c2bbd5b","80c2bbd5d","80c2bbd5f","80c2bbd7","80c2bbdc","80c2bbe4","80c2bbec","80c2bbf1","80c2bbf24",
          "80c2bbf2c","80c2bbf31","80c2bbf3b","80c2bbf3d","80c2bbf3f","80c2bbf6d","80c2bbf73","80c2bbf75","80c2bbf77","80c2bbf85",
          "80c2bbf8b","80c2bbf8d","80c2bbf8f","80c2bbf94","80c2bbf9c","80c2bbfb","80c2bbfd","80c2bbff","80c2bc001","80c2bc007",
          "80c2beaac","80c2beab4","80c2beab9","80c2beabb","80c2bead","80c2beae1","80c2beae7","80c2beb0d","80c2beb0f","80c2beb14",
          "80c2beb1c","80c2beb3","80c2beb5","80c2beb7","80c2beb81","80c2bec7f","80c2bec9","80c2becb","80c2becd","80c2bece4",
          "80c2becec","80c2becf1","80c2becf3","80c2becfb","80c2becfd","80c2bed11","80c2bed15","80c2bed17","80c2bed1c","80c2bed3",
          "80c2bed5","80c2bed67","80c2bed69","80c2bed6b","80c2bf2a4","80c2bf2ac","80c2bf2b4","80c2bf2b9","80c2bf2bb","80c2bf2c5",
          "80c2bf2c7","80c2bf2cc","80c2bf2d4","80c2bf32c","80c2bf333","80c2bf335","80c2bf34b","80c2b870b","80c2b8774","80c2b877c",
          "80c2b879","80c2b87a1","80c2b87a3","80c2b87bd","80c2b87bf","80c2b87c1","80c2b87c3","80c2b87dd","80c2b87df","80c2b87f",
          "80c2b881","80c2b883","80c2b8841","80c2b8847","80c2b8849","80c2b884b","80c2b886b","80c2b886d","80c2b8873","80c2b8875",
          "80c2b898b","80c2b898d","80c2b8993","80c2b8995","80c2b89b5","80c2b89b7","80c2b89b9","80c2b89bf","80c2b89d","80c2b89f",
          "80c2b8a1","80c2b8a21","80c2b8a23","80c2b8a3d","80c2b8a3f","80c2b8a41","80c2b8a43","80c2b8a59","80c2b8a5d","80c2b8a5f",
          "80c2b8a61","80c2b8a65","80c2b8a67","80c2b8a6c","80c2b8a71","80c2b8a73","80c2b8a75","80c2b8af5","80c2b8af7","80c2c7077",
          "80c2c7079","80c2c707d","80c2c707f","80c2c7084","80c2c7089","80c2c708f","80c2c7094","80c2c709c","80c2c70b","80c2c70c1",
          "80c2c70c5","80c2c70c7","80c2c70cc","80c2c70d1","80c2c70d3","80c2c70eb","80c2c75e3","80c2c75f1","80c2c75f5","80c2c75f7",
          "80c2c75fc","80c2c761","80c2c763","80c2c765","80c2c7661","80c2c7665","80c2c7667","80c2c766c","80c2c7673","80c2c76d7",
          "80c2c76d9","80c2c76dd","80c2c76df","80c2c76e4","80c2c76ef","80c2c76f4","80c2c76fc","80c2c774","80c2c77c","80c2c784",
          "80c2c7884","80c2c788c","80c2c7891","80c2c7893","80c2c789b","80c2c789d","80c2c78df","80c2c78f","80c2c791","80c2c7921",
          "80c2c7923","80c2c7939","80c2c793d","80c2c793f","80c2c7944","80c2c795c","80c2c797","80c2c79c","80c2c7a1","80c2c7a3",
          "80c2c7a44","80c2c7a4c","80c2c7a51","80c2c7a5c","80c2c7a7","80c2c7a84","80c2c7a8c","80c2c7a91","80c2c7a9c","80c2c7af5",
          "80c2c7bb5","80c2c7bc5","80c2c7bcc","80c2c7bd4","80c2c7bdc","80c2c7be1","80c2c6144","80c2c6149","80c2c614f","80c2c6151",
          "80c2c6155","80c2c6157","80c2c615c","80c2c617","80c2c619","80c2c61e3","80c2c61e5","80c2c61ed","80c2c61ef","80c2c61f4",
          "80c2c61fc","80c2c624","80c2c629","80c2c62a3","80c2c62b7","80c2c62bc","80c2c62d","80c2c62f","80c2c634","80c2c63c","80c2c6407",
          "80c2c6409","80c2c640b","80c2c645f","80c2c6464","80c2c646f","80c2c6474","80c2c647c","80c2c649","80c2c64b","80c2c64d",
          "80c2c64e1","80c2c64e5","80c2c64e7","80c2c64ec","80c2c64f4","80c2c64fb","80c2c6521","80c2c6525","80c2c6527","80c2c652c",
          "80c2c7ad5","80c2c7b0d","80c2c7b14","80c2c7b23","80c2c7b25","80c2c7b27","80c2c7b2c","80c2c7b34","80c2c7b3c","80c2c7b5",
          "80c2c7b7","80c2c7b84","80c2c7b89","80c2c7b8b","80c2c7b8f","80c2c7b9b","80c2c7b9d","80c2c7b9f","80c2c7bf5","80c2c7c0b",
          "80c2c7c64","80c2c7c6f","80c2c7c74","80c2c7c7c","80c2c7c9","80c2c7cb","80c2c7cd","80c2c7ce1","80c2c7ce5","80c2c7ce7",
          "80c2c7cec","80c2c7cf1","80c2c7cf3","80c2c7cf5","80c2c7d21","80c2c7d25","80c2c7d27","80c2c7d2c","80c2c7d31","80c2c7d33",
          "80c2c8827","80c2c8829","80c2c882b","80c2c89d5","80c2c89d7","80c2c89d9","80c2c89df","80c2c89e1","80c2c89e3","80c2c89fd",
          "80c2c89ff","80c2c8a01","80c2c8a07","80c2c8a09"]
# # Manhattan
# tokens = ["89c258455","89c2584c4","89c2584cf","89c2584d1","89c2584d7","89c2584dc","89c2584e4","89c2584e9","89c2584fb","89c2584fd",
#           "89c2584ff","89c25854","89c25859","89c2585a4","89c2585ac","89c2585b1","89c2585b3","89c2585b7","89c2585bc","89c2585c4",
#           "89c2585c9","89c2585dd","89c2585df","89c2585f","89c258604","89c25860c","89c258611","89c258617","89c25861c","89c258624",
#           "89c258629","89c258813","89c258815","89c258817","89c258819","89c25883","89c25885","89c258864","89c25886c","89c258871",
#           "89c25887b","89c25887d","89c25887f","89c2588c","89c25894","89c2589c","89c258a4","89c258a84","89c258a8c","89c258a91",
#           "89c258a93","89c258a9c","89c258aa1","89c258af4","89c258afc","89c258b04","89c258b09","89c258b0b","89c258b0f","89c258b75",
#           "89c258b85","89c258b87","89c258b8c","89c258b94","89c258b9c","89c258bb","89c258bd","89c258bf","89c258c1","89c258c24",
#           "89c258c2c","89c258c3b","89c258c3d","89c258c3f","89c258c6d","89c258c74","89c258dd5","89c258e13","89c258e15","89c258e17",
#           "89c258e23","89c258e25","89c258e27","89c258e2c","89c258e34","89c258e3c","89c258e5","89c258e64","89c258e6c","89c258e71",
#           "89c258e73","89c258e77","89c258e7c","89c258ec","89c258f4","89c258fc","89c25901","89c25903","89c259041","89c259043",
#           "89c259047","89c25905d","89c25905f","89c25907","89c25909","89c2590b","89c2590cb","89c2590f5","89c2591cd","89c2591d4",
#           "89c2591d9","89c25970b","89c25970d","89c25970f","89c259714","89c259735","89c259737","89c259739","89c25973d","89c25973f",
#           "89c25975","89c25977","89c25979","89c2597a1","89c2597a3","89c2597e3","89c2597e5","89c2597eb","89c2597ed","89c2597ef",
#           "89c2597f4","89c2597fc","89c25984","89c25989","89c2598a4","89c2598ac","89c2598b1","89c2598b3","89c2598b7","89c2598bc",
#           "89c2598c4","89c2598c9","89c2598cf","89c2598d4","89c2598dc","89c2598f","89c25994","89c2599c","89c259a4","89c259ac",
#           "89c259b1","89c259b21","89c259b23","89c259b27","89c259b29","89c259b3d","89c259b3f","89c259b7","89c259bc","89c259c01",
#           "89c259c07","89c259c09","89c259c0b","89c259eab","89c259ead","89c259eb3","89c259eb5","89c259ecc","89c259ed4","89c259f2c",
#           "89c259f55","89c25a10b","89c25a10d","89c25a113","89c25a115","89c25a139","89c25a13f","89c25a141","89c25a143","89c25a147",
#           "89c25a15d","89c25a17","89c25a19","89c25a1a1","89c25a1a3","89c25a1e5","89c25a1ec","89c25a1f4","89c25a1fc","89c25a21",
#           "89c25a224","89c25a22c","89c25a231","89c25a233","89c25a23d","89c25a23f","89c25a244","89c25a249","89c25a24f","89c25a251",
#           "89c25a25c","89c25a27","89c25a29","89c25a2b","89c25a2c1","89c25a2eb","89c25a2ed","89c25a2f4","89c25a3cb","89c25a3cd",
#           "89c25a3cf","89c25a3d4","89c25a3d9","89c25a3db","89c25bd4d","89c25bd4f","89c25bd54","89c25bd59","89c2f5e01","89c2f5e03",
#           "89c2f5e1c","89c2f5e24","89c2f5e2c","89c2f5e31","89c2f5e33","89c2f5e35","89c2f5e3b","89c2f5fcb","89c2f5fcd","89c2f5fcf",
#           "89c2f5fd4","89c2f5fd9","89c2f600d","89c2f600f","89c2f6014","89c2f6019","89c2f601b","89c2f601f","89c2f603","89c2f605",
#           "89c2f607","89c2f60c","89c2f614","89c2f61c","89c2f624","89c2f6284","89c2f628c","89c2f629b","89c2f629d","89c2f62f5",
#           "89c2f62f7","89c2f6395","89c2f63ad","89c2f63b4","89c2f63b9","89c2f63bb","89c2f63bf","89c2f63d","89c2f63e4","89c2f63ec",
#           "89c2f63f1","89c2f63f3","89c2f63f7","89c2f63fc","89c2f6404","89c2f641d","89c2f641f","89c2f662c","89c2f6634","89c2f6639",
#           "89c2f663b","89c2f663f","89c2f665","89c2f6661","89c2f6663","89c2f6667","89c2f6669","89c2f6685","89c2f668c","89c2f6694",
#           "89c2f669c","89c2f66b","89c2f66d","89c2f66f","89c2f674","89c2f67c","89c2f681","89c2f683","89c2f6844","89c2f684c",
#           "89c2f6851","89c2f6853","89c2f6857","89c2f685c","89c2f687","89c2f6881","89c2f6887","89c2f6889","89c2f69ad","89c2f69af",
#           "89c2f69b3","89c2f69b5","89c2f69cb","89c2f69cd","89c2f69d3","89c2f69d5"]



In [ ]:
import s2

def get_s2_children_at_level_recursive(cell_id, target_level):
    """Recursively finds children of an S2 cell at a specific level."""
    if cell_id.level == target_level:
        return [cell_id]
    if cell_id.level > target_level:
        return []

    children = []
    # These methods get the immediate children (at level + 1)
    child_it = cell_id.child_begin()
    child_end_it = cell_id.child_end()

    while child_it != child_end_it:
        children.extend(get_s2_children_at_level_recursive(child_it, target_level))
        child_it = child_it.next()
    return children


def get_s2_children_at_level(s2_cell_tokens, target_level):
  """
  Gets all children of a list of S2 cells at a specific level.

  Args:
    s2_cell_tokens: A list of S2 cell tokens as strings.
    target_level: The desired level for the children cells.

  Returns:
    A dictionary where keys are the parent S2 cell tokens and values are lists
    of their children S2 cell tokens at the target level.
  """
  children_by_parent = {}
  for token in s2_cell_tokens:
    cell_id = s2.S2CellId.from_token(token)
    if cell_id.level >= target_level:
      children_by_parent[token] = [token]
      continue

    child_cells = get_s2_children_at_level_recursive(cell_id, target_level)
    children_by_parent[token] = [c.token() for c in child_cells]
  return children_by_parent

latlngs = []
for s2cell_token in tokens:
  child = get_s2_children_at_level([s2cell_token], 16)[s2cell_token]
  for s in child:
    s2cell_id = s2.S2CellId.from_token(s)
    s2latlng = s2.S2LatLng.from_point(s2.S2Cell(s2cell_id).center())
    latlngs.append((s2latlng.lat.degrees, s2latlng.lng.degrees))
latlngs = jnp.array(latlngs)

In [ ]:
num_latlngs_total = latlngs.shape[0]
num_devices = jax.local_device_count()
batch_size = 32

def _get_reps(train_state, latlngs):
  variables = {'params': train_state.params, **train_state.model_state}
  return model.flax_model.apply(
      variables,
      latlngs=latlngs,
      method=model.flax_model.get_location_embeddings,
  )

pmapped_get_reps = jax.pmap(_get_reps, axis_name='batch')
all_representations = []
for i in range(0, num_latlngs_total, batch_size):
  latlngs_batch = latlngs[i : i + batch_size]
  num_latlngs_batch = latlngs_batch.shape[0]
  pad_amount = (num_devices - (num_latlngs_batch % num_devices)) % num_devices
  if pad_amount > 0:
    padding = jnp.zeros(
        (pad_amount, latlngs_batch.shape[1]), dtype=latlngs_batch.dtype
    )
    latlngs_padded = jnp.concatenate([latlngs_batch, padding], axis=0)
  else:
    latlngs_padded = latlngs_batch

  latlngs_sharded = dataset_utils.shard(latlngs_padded)
  representations_padded = pmapped_get_reps(train_state, latlngs_sharded)
  representations = dataset_utils.unshard(representations_padded)
  all_representations.append(representations[:num_latlngs_batch])
final_reps = jnp.concatenate(all_representations, axis=0)

In [ ]:
rep = {
    'latlng': latlngs,
    'latlng_preds': final_reps,
}

In [ ]:
from scaling_geoloc.viz import vmaps

s2cells = rep['latlng']
plotter = vmaps.GeoPlotter(zoom=10)
num_points = rep['latlng'].shape[0]
mask = np.random.choice(np.arange(rep['latlng'].shape[0]), num_points, replace=False)
plotter.plot(s2cells[mask], color=np.array(misc.features_to_rgb(rep['latlng_preds'][mask], normalize=True, offset=6)), name="PCA")
plotter.fig.show(config={'scrollZoom': True})

If you want to visualize self-similarity of a random point rather than PCA

In [ ]:
repn = final_reps / np.linalg.norm(final_reps, axis=-1, keepdims=True)
repn = repn.astype(np.float32)
plotter = vmaps.GeoPlotter(zoom=11)
idx = np.random.choice(np.arange(repn.shape[0]), 1)
sim = (repn[idx] * repn).sum(-1)
ll = rep['latlng']
topk = jax.lax.approx_max_k(sim, k=repn.shape[0])[1][::-1]

valid = misc.haversine_distance(ll[idx], ll[topk])[..., 0] > 10
topk = topk[valid]

plotter.plot(s2cells[topk], color=sim[topk].astype(np.float32), name="Self-Similarity")
plotter.fig.show(config={'scrollZoom': True})

## tSNE of multimodal embeddings

Given a xid, encode each modality of a batch and compute a tSNE over these representations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE

experiment_config = (211109853, 1)

_, model, train_state = trainer.load_from_xm(*experiment_config)
# train_state = model.replicate_state(train_state)

config = get_config("contrastive_merge")
config.batch_size = 32

dataset = datasets.build_dataset(config.data, config.batch_size, config.batch_size, config.rng_seed)
dummy_batch = next(dataset.valid_iter())

modalities = model.modalities

variables = {'params': train_state.params, **train_state.model_state}
embs = []
modas = []
for modality in modalities:
  if modality == 'aerial':
    embs.append(model.flax_model.apply(
            variables,
            dummy_batch['aerial']['rgb'],
            method=model.flax_model.get_aerial_embeddings,
        ))
  elif modality == 'text':
    embs.append(dummy_batch['siglip_embeddings'])
  elif modality == 'sv':
    embs.append(model.flax_model.apply(
          variables,
          dummy_batch['query']['views']['image_pixels'],
          method=model.flax_model.get_sv_embeddings,
      ))
  elif modality == 'latlng':
    embs.append(model.flax_model.apply(
          variables,
          dummy_batch['query']['views']['latlng'][:, 0],
          method=model.flax_model.get_location_embeddings,
      ))
  elif modality == 'dsm':
    embs.append(model.flax_model.apply(
          variables,
          dummy_batch['dsm']['dsm'],
          method=model.flax_model.get_dsm_embeddings,
      ))
  for _ in range(len(embs[-1])):
    modas.append(modality)

X = jnp.concatenate(embs, axis=0)

tsne = TSNE(n_components=2,
            perplexity=30,
            random_state=42,
            max_iter=3000) # Using max_iter instead of n_iter

X_2d = tsne.fit_transform(X)

tsne_df = {
    'tSNE_1': X_2d[:, 0],
    'tSNE_2': X_2d[:, 1],
    'Modalities': modas
}

plt.figure(figsize=(10, 8))

scatter = sns.scatterplot(
    x='tSNE_1',
    y='tSNE_2',
    hue='Modalities', # Color points based on the target class
    palette=sns.color_palette("bright", len(modalities)), # Use bright colors
    data=tsne_df,
    legend="full",
    alpha=0.8,
    s=100 # Size of the points
)

plt.title('t-SNE Visualization', fontsize=16)
plt.xlabel('t-SNE Component 1', fontsize=12)
plt.ylabel('t-SNE Component 2', fontsize=12)

plt.show()

Pairwise tsne

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from itertools import combinations
import math

experiment_config = (209812324, 1)

_, model, train_state = trainer.load_from_xm(*experiment_config)
# train_state = model.replicate_state(train_state)

config = get_config("contrastive_merge")
config.batch_size = 32

dataset = datasets.build_dataset(config.data, config.batch_size, config.batch_size, config.rng_seed)
dummy_batch = next(dataset.valid_iter())

modalities = model.modalities

variables = {'params': train_state.params, **train_state.model_state}

# 1. Change: Store embeddings in a dictionary for easy pairing
embedding_map = {}

for modality in modalities:
  if modality == 'aerial':
    embedding_map[modality] = model.flax_model.apply(
            variables,
            dummy_batch['aerial']['rgb'],
            method=model.flax_model.get_aerial_embeddings,
        )
  elif modality == 'text':
    embedding_map[modality] = dummy_batch['siglip_embeddings']
  elif modality == 'sv':
    embedding_map[modality] = model.flax_model.apply(
          variables,
          dummy_batch['query']['views']['image_pixels'],
          method=model.flax_model.get_sv_embeddings,
      )
  elif modality == 'latlng':
    embedding_map[modality] = model.flax_model.apply(
          variables,
          dummy_batch['query']['views']['latlng'][:, 0],
          method=model.flax_model.get_location_embeddings,
      )
  elif modality == 'dsm':
    embedding_map[modality] = model.flax_model.apply(
          variables,
          dummy_batch['dsm']['dsm'],
          method=model.flax_model.get_dsm_embeddings,
      )

# 2. Setup the subplot grid
pairs = list(combinations(modalities, 2))
n_plots = len(pairs)
cols = 3
rows = math.ceil(n_plots / cols)

fig, axes = plt.subplots(rows, cols, figsize=(18, 5 * rows))
axes = axes.flatten()

# 3. Loop through pairs and compute t-SNE for each
for i, (mod1, mod2) in enumerate(pairs):
    ax = axes[i]

    emb1 = embedding_map[mod1]
    emb2 = embedding_map[mod2]

    # Concatenate only the two modalities of interest
    X = jnp.concatenate([emb1, emb2], axis=0)

    # Create labels list
    current_labels = [mod1] * len(emb1) + [mod2] * len(emb2)

    # Run t-SNE specific to this pair
    # Note: Perplexity must be < n_samples. If batch is 32, n_samples is 64.
    tsne = TSNE(n_components=2,
                perplexity=min(30, len(current_labels) - 1),
                random_state=42,
                max_iter=1000)

    X_2d = tsne.fit_transform(X)

    tsne_df = {
        'tSNE_1': X_2d[:, 0],
        'tSNE_2': X_2d[:, 1],
        'Modalities': current_labels
    }

    sns.scatterplot(
        x='tSNE_1',
        y='tSNE_2',
        hue='Modalities',
        data=tsne_df,
        ax=ax,
        s=100,
        alpha=0.8
    )

    ax.set_title(f'{mod1} vs {mod2}')
    ax.set_xlabel('') # Remove labels to reduce clutter
    ax.set_ylabel('')

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

## Compute PCA on a grid of latlngs (old)

In [ ]:
start_lat = 40.713
end_lat = 40.717
start_lon = -73.996
end_lon = -73.997

In [ ]:
latlng = []
for i in range (20):
  for j in range (20):
    latlng.append([start_lat + i * (end_lat - start_lat) / 20, start_lon + j * (end_lon - start_lon) / 20])
latlng = np.array(latlng)

In [ ]:
experiment_config = (211109853, 1)

_, model, train_state = trainer.load_from_xm(*experiment_config)
train_state = model.replicate_state(train_state)

In [ ]:
def _get_reps(train_state, latlngs):
  variables = {'params': train_state.params, **train_state.model_state}
  return model.flax_model.apply(
      variables,
      latlngs=latlngs,
      method=model.flax_model.get_location_embeddings,
  )

pmapped_get_reps = jax.pmap(_get_reps, axis_name='batch')
latlngs_sharded = dataset_utils.shard(latlng)
representations_padded = pmapped_get_reps(train_state, latlngs_sharded)
representations = dataset_utils.unshard(representations_padded)

In [ ]:
plotter = vmaps.GeoPlotter(zoom=15)
plotter.plot(latlng, color=np.array(misc.features_to_rgb(representations, normalize=True)), name="PCA")
plotter.fig.show(config={'scrollZoom': True})

# PFDM Evaluation

## Create Dataset with intersecting the one from pdfm and our locations

In [ ]:
with gfile.Open('datasets/conus27.csv', 'rb') as f:
  df_labels = pd.read_csv(f)
latlng = df_labels [['latitude', 'longitude']].values
ids = df_labels ['place'].values
counties = df_labels ['county'].values
df_labels.rename(columns={df_labels.columns[0]: 'id'}, inplace=True)

path = path = 'datasets/area_summaries_sharded/area_summaries/'

s2cells_valid = []
for i in range (2):
  with gfile.Open(path + f'test_grouped_s2cells.csv-{i:05d}-of-00002' , 'rb') as f:
    df = pd.read_csv(f, sep=';')
    s2cells_valid += list(df['s2cell'])

for i in range (32):
  with gfile.Open(path + f'valid_grouped_s2cells.csv-{i:05d}-of-00032' , 'rb') as f:
    df = pd.read_csv(f, sep=';')
    s2cells_valid += list(df['s2cell'])

In [ ]:
import s2

l = []
for s2cell in s2cells_valid:
  list_cell = s2.S2CellId.from_token(s2cell.encode('utf-8'))
  for i in range (5):
    list_cell = list_cell.parent()
  l.append(list_cell.token().decode('utf-8'))

import numpy as np
import bisect # Import the binary search module

def find_indices_in_s2_union(latlng_points: np.ndarray, s2_cell_tokens: list[str]) -> list[int]:
    """
    Efficiently finds the indices of lat/lng points that are inside a set of S2 cells.
    This version uses a binary search on the normalized cell IDs from an S2CellUnion.

    Args:
        latlng_points: A NumPy array of shape (N, 2) with [lat, lng] for N points.
        s2_cell_tokens: A list of S2 cell IDs as string tokens.

    Returns:
        A list of indices of the points contained in the S2 cell union.
    """
    if not s2_cell_tokens or latlng_points.size == 0:
        return []

    # 1. (Pre-computation) Create an S2CellUnion to normalize and sort the cells.
    initial_cell_ids = [s2.S2CellId.from_token(token.encode('utf-8')) for token in s2_cell_tokens]
    s2_union = s2.S2CellUnion(initial_cell_ids)

    # 2. (Optimization) Get the final, sorted, non-overlapping list of cell IDs.
    # --- THIS IS THE CORRECTED LINE ---
    # .cell_ids is an attribute, not a method, so we remove the parentheses.
    union_cell_ids = s2_union.cell_ids

    indices_found = []
    for i, (lat, lng) in enumerate(latlng_points):
        # a. Get the level-30 cell ID for the point.
        point_cell_id = s2.S2CellId.from_lat_lng(s2.S2LatLng.from_degrees(lat, lng))

        # b. Perform a binary search to find the potential parent cell.
        insertion_index = bisect.bisect_right(union_cell_ids, point_cell_id)

        # c. Check if the candidate cell (the one before the insertion point)
        # actually contains our point's cell.
        if insertion_index > 0:
            candidate_cell = union_cell_ids[insertion_index - 1]
            if candidate_cell.contains(point_cell_id):
                indices_found.append(i)

    return indices_found

index_keep = find_indices_in_s2_union(latlng, l)

df_sampled = df_labels.iloc[index_keep].set_index('id')
latlng = df_sampled[['latitude', 'longitude']].values
ids = ids[index_keep]
counties = counties[index_keep]
df_sampled = df_sampled[df_sampled.columns[4:-3]]
keep_mask = [not str(idx).startswith('geoId') for idx in df_sampled.index]
df_sampled = df_sampled[keep_mask]
ids = ids[keep_mask]
counties = counties[keep_mask]
latlng = latlng[keep_mask]

## Dataset vizualisation

In [ ]:
from scaling_geoloc.viz import vmaps

plotter = vmaps.GeoPlotter(zoom=10)
plotter.plot(latlng)
plotter.fig.show(config={'scrollZoom': True})

## Baselines evals

### Geoclip

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
import warnings
from sklearn.exceptions import ConvergenceWarning
from scipy.linalg import LinAlgWarning

with gfile.Open('datasets/pdfm-embeddings/benchmarks/zcta_geoclip.csv', 'rb') as f:
  df_results = pd.read_csv(f)
df_results.set_index('place', inplace=True)
df_final = df_sampled.join(df_results)

X_train_val, X_test = train_test_split(
    df_final, test_size=0.2, random_state=42
)

X_train, X_val = train_test_split(
    X_train_val, test_size=0.125, random_state=42
)

y_train = X_train[X_train.columns[:28]]
X_train = X_train[[f'feature{i}' for i in range (512)]]

y_val = X_val[X_val.columns[:28]]
X_val = X_val[[f'feature{i}' for i in range (512)]]

y_test = X_test[X_test.columns[:28]]
X_test = X_test[[f'feature{i}' for i in range (512)]]

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")

targets = y_train.columns

warnings.filterwarnings(
    action='ignore',
    category=LinAlgWarning,
    message='Ill-conditioned matrix'
)

r2 = []
for target in targets:
  r2_vals = []
  r2_tests = []
  for alpha in [10**(i-1) for i in range(10)]:
    model = Ridge(alpha=alpha)

    is_not_nan = y_train[target].notna()
    model.fit(X_train[is_not_nan], y_train[[target]][is_not_nan])

    is_not_nan_val = y_val[target].notna()
    is_not_nan_test = y_test[target].notna()

    y_pred_val = model.predict(X_val[is_not_nan_val])
    y_pred_test = model.predict(X_test[is_not_nan_test])

    r2_val = r2_score(y_val[[target]][is_not_nan_val], y_pred_val)
    r2_vals.append(r2_val)

    r2_test = r2_score(y_test[[target]][is_not_nan_test], y_pred_test)
    r2_tests.append(r2_test)
  r2_seed = r2_tests[np.argmax(r2_vals)]
  print(f"Target: {target}", r2_seed)
  r2.append(r2_seed)
np.mean(r2)

### Satclip

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
import warnings
from sklearn.exceptions import ConvergenceWarning
from scipy.linalg import LinAlgWarning

with gfile.Open('datasets/pdfm-embeddings/benchmarks/zcta_satclip.csv', 'rb') as f:
  df_results = pd.read_csv(f)
df_results.set_index('place', inplace=True)
df_final = df_sampled.join(df_results)

X_train_val, X_test = train_test_split(
    df_final, test_size=0.2, random_state=42
)

X_train, X_val = train_test_split(
    X_train_val, test_size=0.125, random_state=42
)

y_train = X_train[X_train.columns[:28]]
X_train = X_train[[f'feature{i}' for i in range (256)]]

y_val = X_val[X_val.columns[:28]]
X_val = X_val[[f'feature{i}' for i in range (256)]]

y_test = X_test[X_test.columns[:28]]
X_test = X_test[[f'feature{i}' for i in range (256)]]

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")

targets = y_train.columns

warnings.filterwarnings(
    action='ignore',
    category=LinAlgWarning,
    message='Ill-conditioned matrix'
)

r2 = []
for target in targets:
  r2_vals = []
  r2_tests = []
  for alpha in [10**(i-1) for i in range(10)]:
    model = Ridge(alpha=alpha)

    is_not_nan = y_train[target].notna()
    model.fit(X_train[is_not_nan], y_train[[target]][is_not_nan])

    is_not_nan_val = y_val[target].notna()
    is_not_nan_test = y_test[target].notna()

    y_pred_val = model.predict(X_val[is_not_nan_val])
    y_pred_test = model.predict(X_test[is_not_nan_test])

    r2_val = r2_score(y_val[[target]][is_not_nan_val], y_pred_val)
    r2_vals.append(r2_val)

    r2_test = r2_score(y_test[[target]][is_not_nan_test], y_pred_test)
    r2_tests.append(r2_test)
  r2_seed = r2_tests[np.argmax(r2_vals)]
  print(f"Target: {target}", r2_seed)
  r2.append(r2_seed)
np.mean(r2)

### PDFM

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score
import warnings
from sklearn.exceptions import ConvergenceWarning
from scipy.linalg import LinAlgWarning

with gfile.Open('datasets/pdfm-embeddings/v0/us/zcta_embeddings.csv', 'rb') as f:
  df_results = pd.read_csv(f)
df_results.set_index('place', inplace=True)
df_final = df_sampled.join(df_results)

X_train_val, X_test = train_test_split(
    df_final, test_size=0.2, random_state=42
)

X_train, X_val = train_test_split(
    X_train_val, test_size=0.125, random_state=42
)

y_train = X_train[X_train.columns[:28]]
X_train = X_train[[f'feature{i}' for i in range (330)]]

y_val = X_val[X_val.columns[:28]]
X_val = X_val[[f'feature{i}' for i in range (330)]]

y_test = X_test[X_test.columns[:28]]
X_test = X_test[[f'feature{i}' for i in range (330)]]

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")

targets = y_train.columns

warnings.filterwarnings(
    action='ignore',
    category=LinAlgWarning,
    message='Ill-conditioned matrix'
)

r2 = []
for target in targets:
  r2_vals = []
  r2_tests = []
  for alpha in [10**(i-1) for i in range(10)]:
    model = Ridge(alpha=alpha)
    # model = LinearRegression()

    is_not_nan = y_train[target].notna()
    model.fit(X_train[is_not_nan], y_train[[target]][is_not_nan])

    is_not_nan_val = y_val[target].notna()
    is_not_nan_test = y_test[target].notna()

    y_pred_val = model.predict(X_val[is_not_nan_val])
    y_pred_test = model.predict(X_test[is_not_nan_test])

    r2_val = r2_score(y_val[[target]][is_not_nan_val], y_pred_val)
    r2_vals.append(r2_val)

    r2_test = r2_score(y_test[[target]][is_not_nan_test], y_pred_test)
    r2_tests.append(r2_test)
  r2_seed = r2_tests[np.argmax(r2_vals)]
  print(f"Target: {target}", r2_seed)
  r2.append(r2_seed)
np.mean(r2)

### Siglip

Load csv that mmaps us zip codes to name of city, county, state

In [ ]:
from colabtools import drive
import io
import pandas as pd

raw_data = drive.LoadFile(file_id='10xLH-N93CUYU5_HShHFsgGJWpeaIk0L_')
data_io = io.BytesIO(raw_data)
df = pd.read_csv(data_io)

In [ ]:
cities = []
states = []
counties = []
for id in ids:
  if int(id.split('/')[-1]) in df['zip'].values:
    cities.append(df[df['zip'] == int(id.split('/')[-1])]['city'].values[0])
    states.append(df[df['zip'] == int(id.split('/')[-1])]['state'].values[0])
    counties.append(df[df['zip'] == int(id.split('/')[-1])]['county'].values[0])
  else:
    cities.append('')
    states.append('')
    counties.append('')

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import tensorflow as tf

# --- Imports (Identiques) ---
import big_vision.pp.builder as pp_builder
import big_vision.pp.ops_text  # pylint: disable=unused-import
import big_vision.pp.proj.paligemma.ops  # pylint: disable=unused-import
from unigeoclip.scenic.models import siglip_text
from big_vision.models.proj.image_text import text_transformer
from big_vision.models.proj.image_text import two_towers

# --- 1. Configuration TPU (Simplifiée) ---
print("--- Configuration TPU ---")
num_devices = jax.device_count()
if num_devices == 0:
    raise ValueError("Aucun appareil JAX (TPU) trouvé.")

# Taille du batch GLOBAL (sera divisée par le nombre de TPUs)
GLOBAL_BATCH_SIZE = 64 * num_devices
TOKEN_LENGTH = 64

print(f"Exécution sur {num_devices} appareils.")
print(f"Taille de batch globale: {GLOBAL_BATCH_SIZE}")


# --- 2. Modèle, Paramètres et Prétraitement (Identiques) ---
print("--- Initialisation du modèle et du prétraitement ---")
pp_txt = pp_builder.get_preprocess_fn(
    f'lower(key="text")|tok(length={TOKEN_LENGTH}, model="gemma",'
    ' bos="no", eos="sticky", key="text")'
)

# model = siglip_text.PreTrainedSigLip2TextModel(
#     variant='So400m',
#     output_dim=1152,
#     vocab_size=256000,
#     model_name='SigLIP2 So400m/16 256',
# )

model = siglip_text.PreTrainedSigLip2TextModel(
    variant='B',
    output_dim=768,
    vocab_size=256000,
    model_name='SigLIP2 B/16 256',
)

dummy_inputs = np.zeros((1, TOKEN_LENGTH), np.int32)
variables = model.init(jax.random.PRNGKey(0), dummy_inputs)

init_files = two_towers.VANITY_NAMES.get(
    model.model_name, model.model_name
)
init_files = {k: f'{init_files}:{k}' for k in ('img', 'txt', 't')}
txt_init = init_files.pop('text', init_files.pop('txt', None))

print("Chargement des paramètres pré-entraînés...")
loaded_model_params = text_transformer.load(
    variables['params']['model'], txt_init, {}
)
params = {'params': {'model': loaded_model_params}}


# --- 3. Parallélisation du Modèle (Identique) ---
print("--- Parallélisation du modèle pour TPU ---")

# La fonction JAX pure à paralléliser
@jax.jit
def inference_step(params, batch_tokens):
    return model.apply(params, batch_tokens, train=False)

# Créer une version parallèle avec pmap
p_inference_step = jax.pmap(inference_step, axis_name='batch')

# Répliquer les paramètres du modèle sur tous les TPUs (une seule fois)
p_params = jax.device_put_replicated(params, jax.local_devices())
print("Paramètres du modèle répliqués sur tous les appareils.")


# --- 4. Données (Exemple de données) ---

# Créer la liste des prompts
# texts = [f"{city} city in the US" for i, city in enumerate(cities)]
texts = [f"{city} city in {counties[i]} county in {states[i]} state in the US" for i, city in enumerate(cities)] # augmented
print(f"Total de {len(texts)} textes à traiter.")


# --- 5. Boucle d'Inférence par Batch (Version simplifiée) ---
print("--- Démarrage de l'inférence par batch ---")
all_embeddings = []

# Calcule le nombre total de batches
total_batches = int(np.ceil(len(texts) / GLOBAL_BATCH_SIZE))

for i in range(total_batches):
    # 1. Obtenir le batch de textes
    start_idx = i * GLOBAL_BATCH_SIZE
    end_idx = start_idx + GLOBAL_BATCH_SIZE
    batch_texts = texts[start_idx:end_idx]

    # 2. Tokeniser CE batch (sur CPU)
    # C'est la ligne qui évite l'erreur "Out of Memory"
    batch_tokens = np.array([pp_txt({'text': t})['text'] for t in batch_texts])

    # 3. Gérer le dernier batch (padding)
    actual_size = batch_tokens.shape[0]
    if actual_size < GLOBAL_BATCH_SIZE:
        padded_batch = np.zeros((GLOBAL_BATCH_SIZE, TOKEN_LENGTH), dtype=np.int32)
        padded_batch[:actual_size] = batch_tokens
        batch_to_run = padded_batch
    else:
        batch_to_run = batch_tokens

    # 4. Diviser le batch pour les TPUs
    # (Global_BS, 64) -> (Num_Appareils, Batch_Par_Appareil, 64)
    sharded_batch = batch_to_run.reshape((num_devices, -1, TOKEN_LENGTH))

    # 5. Exécuter l'inférence en parallèle
    batch_embeds_sharded = p_inference_step(p_params, sharded_batch)

    # 6. Rassembler les résultats de tous les TPUs
    batch_embeds = jax.device_get(batch_embeds_sharded)
    batch_embeds = batch_embeds.reshape(GLOBAL_BATCH_SIZE, -1)

    # 7. Retirer le padding
    if actual_size < GLOBAL_BATCH_SIZE:
        batch_embeds = batch_embeds[:actual_size]

    all_embeddings.append(batch_embeds)

    if (i + 1) % 10 == 0 or i == total_batches - 1:
        print(f"  ... batch {i + 1} / {total_batches} traité.")

print("Inférence terminée.")

# --- 6. Post-traitement (Identique) ---
print("--- Post-traitement vers DataFrame ---")

# Concaténer tous les résultats des batches
text_embed = np.concatenate(all_embeddings, axis=0)

size = text_embed.shape[-1]

print(f"Forme finale des embeddings: {text_embed.shape}")

df_results = {}
df_results['id'] = ids
df_results['latitude'] = latlng[:, 0]
df_results['longitude'] = latlng[:, 1]

embedding_dim = text_embed.shape[1]
for i in range(embedding_dim):
  df_results[f'rep_{i}'] = text_embed[:, i]

df_results = pd.DataFrame(df_results).set_index('id')

print("DataFrame créé avec succès.")
df_final = df_sampled.join(df_results)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
import warnings
from sklearn.exceptions import ConvergenceWarning
from scipy.linalg import LinAlgWarning

X_train_val, X_test = train_test_split(
    df_final, test_size=0.2, random_state=42
)

X_train, X_val = train_test_split(
    X_train_val, test_size=0.125, random_state=42
)

y_train = X_train[X_train.columns[:28]]
X_train = X_train[[f'rep_{i}' for i in range (size)]]

y_val = X_val[X_val.columns[:28]]
X_val = X_val[[f'rep_{i}' for i in range (size)]]

y_test = X_test[X_test.columns[:28]]
X_test = X_test[[f'rep_{i}' for i in range (size)]]

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")

targets = y_train.columns

warnings.filterwarnings(
    action='ignore',
    category=LinAlgWarning,
    message='Ill-conditioned matrix'
)

r2 = []
for target in targets:
  r2_vals = []
  r2_tests = []
  for alpha in [10**(i-1) for i in range(10)]:
    model = Ridge(alpha=alpha)

    is_not_nan = y_train[target].notna()
    model.fit(X_train[is_not_nan], y_train[[target]][is_not_nan])

    is_not_nan_val = y_val[target].notna()
    is_not_nan_test = y_test[target].notna()

    y_pred_val = model.predict(X_val[is_not_nan_val])
    y_pred_test = model.predict(X_test[is_not_nan_test])

    r2_val = r2_score(y_val[[target]][is_not_nan_val], y_pred_val)
    r2_vals.append(r2_val)

    r2_test = r2_score(y_test[[target]][is_not_nan_test], y_pred_test)
    r2_tests.append(r2_test)
  r2_seed = r2_tests[np.argmax(r2_vals)]
  print(f"Target: {target}", r2_seed)
  r2.append(r2_seed)
np.mean(r2)

## Evals of our models

Given a xid, returns the score on pdfm with the detailed scorese

In [ ]:
experiment_config = (211218120, 1)

_, model, train_state = trainer.load_from_xm(*experiment_config)
train_state = model.replicate_state(train_state)

num_latlngs_total = latlng.shape[0]
num_devices = jax.local_device_count()
batch_size = 32

def _get_reps(train_state, latlngs):
  variables = {'params': train_state.params, **train_state.model_state}
  return model.flax_model.apply(
      variables,
      latlngs=latlngs,
      # proj=False,
      method=model.flax_model.get_location_embeddings,
  )

pmapped_get_reps = jax.pmap(_get_reps, axis_name='batch')
all_representations = []
for i in range(0, num_latlngs_total, batch_size):
  latlngs_batch = latlng[i : i + batch_size]
  num_latlngs_batch = latlngs_batch.shape[0]
  pad_amount = (num_devices - (num_latlngs_batch % num_devices)) % num_devices
  if pad_amount > 0:
    padding = jnp.zeros(
        (pad_amount, latlngs_batch.shape[1]), dtype=latlngs_batch.dtype
    )
    latlngs_padded = jnp.concatenate([latlngs_batch, padding], axis=0)
  else:
    latlngs_padded = latlngs_batch

  latlngs_sharded = dataset_utils.shard(latlngs_padded)
  representations_padded = pmapped_get_reps(train_state, latlngs_sharded)
  representations = dataset_utils.unshard(representations_padded)
  all_representations.append(representations[:num_latlngs_batch])
final_reps = jnp.concatenate(all_representations, axis=0)

df_results = {}
df_results['id'] = ids
df_results['latitude'] = latlng[:, 0]
df_results['longitude'] = latlng[:, 1]
for i in range(final_reps.shape[1]):
  df_results[f'rep_{i}'] = final_reps[:, i]
df_results = pd.DataFrame(df_results).set_index('id')
df_final = df_sampled.join(df_results)

from sklearn.model_selection import train_test_split

X_train_val, X_test = train_test_split(
    df_final, test_size=0.2, random_state=42
)

X_train, X_val = train_test_split(
    X_train_val, test_size=0.125, random_state=42
)

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")

y_train = X_train[X_train.columns[:28]]
X_train = X_train[[f'rep_{i}' for i in range (768)]]

y_val = X_val[X_val.columns[:28]]
X_val = X_val[[f'rep_{i}' for i in range (768)]]

y_test = X_test[X_test.columns[:28]]
X_test = X_test[[f'rep_{i}' for i in range (768)]]

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
import warnings
from scipy.linalg import LinAlgWarning

targets = y_train.columns[1:]

warnings.filterwarnings(
    action='ignore',
    category=LinAlgWarning,
    message='Ill-conditioned matrix'
)

r2 = []
for target in targets:
  r2_vals = []
  r2_tests = []
  for alpha in [10**(i-1) for i in range(10)]:
    model_r = Ridge(alpha=alpha)

    is_not_nan = y_train[target].notna()
    model_r.fit(X_train[is_not_nan], y_train[[target]][is_not_nan])

    is_not_nan_val = y_val[target].notna()
    is_not_nan_test = y_test[target].notna()

    y_pred_val = model_r.predict(X_val[is_not_nan_val])
    y_pred_test = model_r.predict(X_test[is_not_nan_test])

    r2_val = r2_score(y_val[[target]][is_not_nan_val], y_pred_val)
    r2_vals.append(r2_val)

    r2_test = r2_score(y_test[[target]][is_not_nan_test], y_pred_test)
    r2_tests.append(r2_test)
  r2_seed = r2_tests[np.argmax(r2_vals)]
  print(f"Target: {target}", r2_seed)
  r2.append(r2_seed)
np.mean(r2)

### Eval with text

In [ ]:
from colabtools import drive
import io
import pandas as pd

raw_data = drive.LoadFile(file_id='10xLH-N93CUYU5_HShHFsgGJWpeaIk0L_')
data_io = io.BytesIO(raw_data)
df = pd.read_csv(data_io)

In [ ]:
cities = []
states = []
counties = []
for id in ids:
  if int(id.split('/')[-1]) in df['zip'].values:
    cities.append(df[df['zip'] == int(id.split('/')[-1])]['city'].values[0])
    states.append(df[df['zip'] == int(id.split('/')[-1])]['state'].values[0])
    counties.append(df[df['zip'] == int(id.split('/')[-1])]['county'].values[0])
  else:
    cities.append('')
    states.append('')
    counties.append('')

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import tensorflow as tf

# --- Imports (Identiques) ---
import big_vision.pp.builder as pp_builder
import big_vision.pp.ops_text  # pylint: disable=unused-import
import big_vision.pp.proj.paligemma.ops  # pylint: disable=unused-import
from unigeoclip.scenic.models import siglip_text
from big_vision.models.proj.image_text import text_transformer
from big_vision.models.proj.image_text import two_towers

pp_txt = pp_builder.get_preprocess_fn(
    f'lower(key="text")|tok(length={64}, model="gemma",'
    ' bos="no", eos="sticky", key="text")'
)

texts = [f"{city} city in {counties[i]} county in {states[i]} state in the US" for i, city in enumerate(cities)] # augmented

texts_tokenized = jnp.array([pp_txt({'text': text})['text'] for text in texts])

train_state = model.unreplicate_state(train_state)

num_texts_total = texts_tokenized.shape[0]
num_devices = jax.local_device_count()
batch_size = 32

def _get_reps(train_state, texts):
  variables = {'params': train_state.params, **train_state.model_state}
  return model.flax_model.apply(
      variables,
      text=texts,
      # proj=False,
      method=model.flax_model.get_text_embeddings,
  )

pmapped_get_reps = jax.pmap(_get_reps, axis_name='batch')
all_representations = []
for i in range(0, num_texts_total, batch_size):
  latlngs_batch = texts_tokenized[i : i + batch_size]
  num_latlngs_batch = latlngs_batch.shape[0]
  pad_amount = (num_devices - (num_latlngs_batch % num_devices)) % num_devices
  if pad_amount > 0:
    padding = jnp.zeros(
        (pad_amount, latlngs_batch.shape[1]), dtype=latlngs_batch.dtype
    )
    latlngs_padded = jnp.concatenate([latlngs_batch, padding], axis=0)
  else:
    latlngs_padded = latlngs_batch

  latlngs_sharded = dataset_utils.shard(latlngs_padded)
  representations_padded = pmapped_get_reps(train_state, latlngs_sharded)
  representations = dataset_utils.unshard(representations_padded)
  all_representations.append(representations[:num_latlngs_batch])
final_reps_text = jnp.concatenate(all_representations, axis=0)

In [ ]:
final_reps = np.concatenate([final_reps_text, final_reps], axis=1)

df_results = {}
df_results['id'] = ids
df_results['latitude'] = latlng[:, 0]
df_results['longitude'] = latlng[:, 1]
for i in range(final_reps.shape[1]):
  df_results[f'rep_{i}'] = final_reps[:, i]
df_results = pd.DataFrame(df_results).set_index('id')
df_final = df_sampled.join(df_results)

from sklearn.model_selection import train_test_split

X_train_val, X_test = train_test_split(
    df_final, test_size=0.2, random_state=42
)

X_train, X_val = train_test_split(
    X_train_val, test_size=0.125, random_state=42
)

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")

y_train = X_train[X_train.columns[:28]]
X_train = X_train[[f'rep_{i}' for i in range (768 * 2)]]

y_val = X_val[X_val.columns[:28]]
X_val = X_val[[f'rep_{i}' for i in range (768 * 2)]]

y_test = X_test[X_test.columns[:28]]
X_test = X_test[[f'rep_{i}' for i in range (768 * 2)]]

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
import warnings
from scipy.linalg import LinAlgWarning

targets = y_train.columns[1:]

warnings.filterwarnings(
    action='ignore',
    category=LinAlgWarning,
    message='Ill-conditioned matrix'
)

r2 = []
for target in targets:
  r2_vals = []
  r2_tests = []
  for alpha in [10**(i-1) for i in range(10)]:
    model_r = Ridge(alpha=alpha)

    is_not_nan = y_train[target].notna()
    model_r.fit(X_train[is_not_nan], y_train[[target]][is_not_nan])

    is_not_nan_val = y_val[target].notna()
    is_not_nan_test = y_test[target].notna()

    y_pred_val = model_r.predict(X_val[is_not_nan_val])
    y_pred_test = model_r.predict(X_test[is_not_nan_test])

    r2_val = r2_score(y_val[[target]][is_not_nan_val], y_pred_val)
    r2_vals.append(r2_val)

    r2_test = r2_score(y_test[[target]][is_not_nan_test], y_pred_test)
    r2_tests.append(r2_test)
  r2_seed = r2_tests[np.argmax(r2_vals)]
  print(f"Target: {target}", r2_seed)
  r2.append(r2_seed)
np.mean(r2)

# Pv4Ger Evaluation (classif)

From a xid either run the evalution from script either generates the dataset embedded

In [ ]:
experiment_config = (211109853, 1)

eval_name = 'pv4ger'

_, model, train_state = trainer.load_from_xm(*experiment_config)
train_state = model.replicate_state(train_state)

config = get_config(config_name="eval_geolocation_retrieval")
eval_config = config.test_evals[eval_name]

evaluator = load.get_eval(eval_config.get('eval_name', eval_name))(
      model,
      eval_config,
  )

train, valid = evaluator.get_database_representations(train_state)
# _, metrics, _ = evaluator.evaluate(train_state)
# print(metrics)

## Zero shot classification

In [ ]:
import big_vision.pp.builder as pp_builder
import big_vision.pp.ops_text  # pylint: disable=unused-import
import big_vision.pp.proj.paligemma.ops  # pylint: disable=unused-import

variables = {'params': jax_utils.unreplicate(train_state.params), **train_state.model_state}
pp_txt = pp_builder.get_preprocess_fn(
        f'lower(key="text")|tok(length={64}, model="gemma",'
        ' bos="no", eos="sticky", key="text")'
    )

texts = ["an aerial view of solar panels", "an aerial view of something else"]
# texts = ["an aerial view of solar panels.", "an aerial view wihtout any solar panel"]
texts_tokenized = jnp.array([pp_txt({'text': text})['text'] for text in texts])

variables = {'params': train_state.params, **train_state.model_state}
text_embed = model.flax_model.apply(variables, texts_tokenized, method=model.flax_model.get_text_embeddings)

In [ ]:
similarities = np.einsum('bd,cd->bc', valid['embedding']/ np.linalg.norm(valid['embedding'], axis=1, keepdims=True),
                         text_embed / np.linalg.norm(text_embed, axis=1, keepdims=True))
y_pred = np.argmax(similarities, axis=1)
y_pred[y_pred > 1] = 1
y_pred = 1 - y_pred
print(np.mean(y_pred == valid['label']), np.mean(y_pred), np.mean(valid['label']))

In [ ]:
np.mean(y_pred), np.mean(valid['label'])

## Linear Probing

In [ ]:
mean = np.mean(train['embedding'], axis=0)
std = np.std(train['embedding'], axis=0)
train_embed = (train['embedding'] - mean) / std
valid_embed = (valid['embedding'] - mean) / std

In [ ]:
import tensorflow as tf
from tensorflow.keras import optimizers

lrs = [0.02, 0.01, 0.005, 0.002, 0.001, 0.0005, 0.0002]
# lr = 0.002
# batch_sizes = [4,8,16,32, 64, 128,256,512]
# batch_sizes = [1024]
bs = 2048
for lr in lrs:
# for bs in batch_sizes:
  # optim = optimizers.SGD(learning_rate=lr)
  optim = optimizers.AdamW(learning_rate=lr)

  # Compile the model for binary classification
  module_list = []
  # Output layer.
  module_list.append(tf.keras.layers.Dense(1, activation="sigmoid"))
  model = tf.keras.Sequential(module_list)

  model.compile(
      optimizer=optim,
      loss="binary_crossentropy",
      metrics=['accuracy'],
  )

  history = model.fit(
      train_embed,
      train['label'],
      epochs=500,
      batch_size=bs,
      validation_data=(valid_embed, valid['label']),
      verbose=False,
  )
  probabilities = model.predict(valid_embed, verbose=False)
  predicted_classes = (probabilities > 0.5).astype(int)
  print(lr, round(np.mean(predicted_classes.squeeze() == valid['label']), 3))

In [ ]:
history_dict = history.history

train_loss = history_dict['loss']
train_accuracy = history_dict['accuracy']
val_loss = history_dict['val_loss']
val_accuracy = history_dict['val_accuracy']

# Create a figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Plot training & validation loss values
ax1.plot(train_loss, label='Training Loss')
ax1.plot(val_loss, label='Validation Loss')
ax1.set_title('Model Loss')
ax1.set_ylabel('Loss')
ax1.set_xlabel('Epoch')
ax1.legend(loc='upper right')
ax1.grid(True)


# Plot training & validation accuracy values
ax2.plot(train_accuracy, label='Training Accuracy')
ax2.plot(val_accuracy, label='Validation Accuracy')
ax2.set_title('Model Accuracy')
ax2.set_ylabel('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend(loc='lower right')
ax2.grid(True)

plt.show()

## Siglip evaluation

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

# Original imports
from unigeoclip.scenic.models import siglip
from unigeoclip.scenic import datasets

# --- 1. Model and Config Setup (Mostly Unchanged) ---
model_name = 'SigLIP2 B/16 256' # or 'SigLIP2 So400m/16 256'
model_name = 'SigLIP2 So400m/16 256'
variant = 'So400m' if model_name == 'SigLIP2 So400m/16 256' else 'B'

print("Initializing model...")
model = siglip.PreTrainedSigLip2Model(
    variant='So400m',
    pool_type='map',
    model_name=model_name,
)

# Dummy init remains the same
dummy_inputs = np.zeros((1, 256, 256, 3), np.float32)
variables = model.init(jax.random.PRNGKey(0), dummy_inputs)

print("Loading pretrained parameters...")
loaded_model_params = model.load_pretrained_params(model_name)
params = {'params': loaded_model_params}

config = get_config()
eval_name = 'pv4ger'
eval_config = config.test_evals[eval_name]

# --- 2. Configure for Parallel TPU Execution ---

num_devices = jax.device_count()
if num_devices == 0:
    raise ValueError("No JAX devices found. This optimization requires TPUs.")

# Assume the config's batch_size is the GLOBAL batch size.
# It must be divisible by the number of devices.
global_batch_size = 128
if global_batch_size % num_devices != 0:
    print(
        f"Warning: Global batch size ({global_batch_size}) not divisible "
        f"by device count ({num_devices}). Adjusting global batch size."
    )
    # Adjust batch size to be a multiple of num_devices
    global_batch_size = (global_batch_size // num_devices) * num_devices
    if global_batch_size == 0:
         # Fallback if original batch_size < num_devices
         global_batch_size = num_devices
         print(f"Setting global batch size to {global_batch_size}.")

eval_config.batch_size_eval = global_batch_size
eval_config.batch_size = global_batch_size

print(f"Running on {num_devices} devices.")
print(f"Global batch size: {global_batch_size}")
print(f"Per-device batch size: {global_batch_size // num_devices}")

# Build the dataset. The dataset pipeline should automatically handle
# sharding the global_batch_size across devices.
dataset = datasets.build_dataset(
    eval_config.data,
    eval_config.batch_size,       # train_batch_size (global)
    eval_config.batch_size_eval,  # eval_batch_size (global)
    0
)

# --- 3. Define and Parallelize the Inference Step ---

def inference_step(params, batch):
  """
  A single inference step for one device's shard of data.
  This function will be JIT-compiled and run in parallel.
  """
  inputs = batch['input']

  # Resize operation (now JIT-compiled as part of the step)
  resized_inputs = jax.image.resize(
      inputs,
      (inputs.shape[0], 256, 256, 3),  # inputs.shape[0] is per-device batch size
      jax.image.ResizeMethod.LINEAR,
  )

  # Model application
  embeddings = model.apply(params, resized_inputs, train=False)[0]

  return embeddings, batch['label']

# Use jax.pmap to create a parallel version of the inference step.
# This compiles the function and distributes it to all TPU cores.
p_inference_step = jax.pmap(inference_step, axis_name='batch')

# --- 4. Replicate Parameters to All Devices ---

print("Replicating model parameters to all devices...")
# This copies the model weights to each TPU core's memory.
p_params = jax.device_put_replicated(params, jax.local_devices())
print("Replication complete.")

# --- 5. Refactored Feature Extraction Loop ---

def extract_features(iterator, p_inference_fn):
    """
    Runs feature extraction over a dataset iterator using the
    parallelized inference function.
    """
    labels, embed = [], []
    print("Starting feature extraction... (First batch will trigger JIT compilation)")

    for i, batch in enumerate(iterator):
        # batch is already sharded by the data loader:
        # Shape: (num_devices, per_device_batch_size, H, W, 3)

        actual_size = batch['input'].shape[0]

        # Handle padding if batch size is not divisible by num_devices
        # We pad to the next multiple of num_devices, or just global_batch_size if smaller
        remainder = actual_size % num_devices
        if remainder != 0:
            pad_amt = num_devices - remainder
            # We need to pad all arrays in the batch dict
            padded_batch = {}
            for k, v in batch.items():
                # Assume v is (B, ...)
                pad_shape = (pad_amt,) + v.shape[1:]
                padding = np.zeros(pad_shape, dtype=v.dtype)
                padded_batch[k] = np.concatenate([v, padding], axis=0)
            batch_to_shard = padded_batch
        else:
            batch_to_shard = batch

        # Run inference in parallel across all devices
        batch_sharded = dataset_utils.shard(batch_to_shard)
        # print(batch_sharded['input'].shape)
        batch_embeds, batch_labels = p_inference_fn(p_params, batch_sharded)

        # Gather results from all devices back to the host CPU
        batch_embeds = jax.device_get(batch_embeds)
        batch_labels = jax.device_get(batch_labels)

        # Reshape from (num_devices, per_device_bs, ...) to (total_bs, ...)
        batch_embeds = batch_embeds.reshape(-1, *batch_embeds.shape[2:])
        batch_labels = batch_labels.reshape(-1, *batch_labels.shape[2:])

        # Remove padding if we padded
        if remainder != 0:
            batch_embeds = batch_embeds[:actual_size]
            batch_labels = batch_labels[:actual_size]

        labels.append(batch_labels)
        embed.append(batch_embeds)

        if (i + 1) % 50 == 0:
            print(f"  ... processed {i + 1} batches.")

    print("Feature extraction complete.")

    if not labels:
        print("Warning: No data found in iterator.")
        return np.array([]), np.array([])

    # Concatenate all batches into two large numpy arrays
    return np.concatenate(labels), np.concatenate(embed)

# --- 6. Run Extraction ---

print("\n--- Extracting Training Set ---")
train_label, train_embed = extract_features(dataset.train_iter(), p_inference_step)
print(f"Training set extracted: embeds {train_embed.shape}, labels {train_label.shape}")

print("\n--- Extracting Validation Set ---")
valid_label, valid_embed = extract_features(dataset.valid_iter(), p_inference_step)
print(f"Validation set extracted: embeds {valid_embed.shape}, labels {valid_label.shape}")

print("\nAll features extracted.")

In [ ]:
train_embed.shape, train_label.shape, valid_embed.shape, valid_label.shape

In [ ]:
similarities = np.einsum('qe,de->qd', valid_embed / np.linalg.norm(valid_embed, axis=1, keepdims=True),
                         train_embed / np.linalg.norm(train_embed, axis=1, keepdims=True))

for k in [1, 5, 10]:
  top_k_indices = np.argsort(similarities, axis=-1)[..., -k:]
  top_k_labels = train_label[top_k_indices]

  # Majority vote.
  preds = np.apply_along_axis(
      lambda x: np.bincount(x).argmax(), 1, top_k_labels
  )
  accuracy = np.mean(preds == valid_label.squeeze())
  print(f'knn_accuracy@{k}:', accuracy * 100.0)

In [ ]:
import big_vision.pp.builder as pp_builder
import big_vision.pp.ops_text  # pylint: disable=unused-import
import big_vision.pp.proj.paligemma.ops  # pylint: disable=unused-import
from unigeoclip.scenic.models import siglip_text
from big_vision.models.proj.image_text import text_transformer
from big_vision.models.proj.image_text import two_towers

pp_txt = pp_builder.get_preprocess_fn(
        f'lower(key="text")|tok(length={64}, model="gemma",'
        ' bos="no", eos="sticky", key="text")'
    )

# model = siglip_text.PreTrainedSigLip2TextModel(
#     variant='So400m',
#     output_dim=1152,
#     vocab_size=256000,
#     model_name='SigLIP2 So400m/16 256',
# )

model = siglip_text.PreTrainedSigLip2TextModel(
    variant='B',
    output_dim=768,
    vocab_size=256000,
    model_name='SigLIP2 B/16 256',
)

dummy_inputs = np.zeros((1, 64), np.int32)
variables = model.init(jax.random.PRNGKey(0), dummy_inputs)

# Get checkpoint path for text tower
init_files = two_towers.VANITY_NAMES.get(
    model.model_name, model.model_name
)
init_files = {k: f'{init_files}:{k}' for k in ('img', 'txt', 't')}
txt_init = init_files.pop('text', init_files.pop('txt', None))

loaded_model_params = text_transformer.load(
    variables['params']['model'], txt_init, {}
)
params = {'params': {'model': loaded_model_params}}

In [ ]:
texts = ["an aerial view of solar panels", "an aerial view of something else"]
#["an aerial view of solar panels", "an aerial view not containing solar panels"]
texts_tokenized = jnp.array([pp_txt({'text': text})['text'] for text in texts])

text_embed = model.apply(params, texts_tokenized, train=False)

similarities = np.einsum('bd,cd->bc', valid_embed, text_embed / np.linalg.norm(text_embed, axis=1, keepdims=True))
y_pred = np.argmax(similarities, axis=1)
print(np.mean(similarities, axis=0))
y_pred[y_pred > 1] = 1
y_pred = 1 - y_pred
print(round(np.mean(y_pred == valid_label), 3), round(np.mean(y_pred), 3), round(np.mean(valid_label), 3))

In [ ]:
valid_embed.shape

In [ ]:
import tensorflow as tf
from tensorflow.keras import optimizers

# lrs = [0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001, 0.0005, 0.0002, 0.0001]
lr = 0.002
# batch_sizes = [4,8,16,32, 64, 128,256,512]
batch_sizes = [128]
# for lr in lrs:
for bs in batch_sizes:
  optim = optimizers.SGD(learning_rate=lr)

  # Compile the model for binary classification
  module_list = []
  # Output layer.
  module_list.append(tf.keras.layers.Dense(1, activation="sigmoid"))
  model = tf.keras.Sequential(module_list)

  model.compile(
      optimizer=optim,
      loss="binary_crossentropy",
      metrics=['accuracy'],
  )

  history = model.fit(
      train_embed,
      train_label,
      epochs=200,
      batch_size=bs,
      validation_data=(valid_embed, valid_label),
      verbose=True,
  )
  probabilities = model.predict(valid_embed, verbose=False)
  predicted_classes = (probabilities > 0.5).astype(int)
  print(round(np.mean(predicted_classes.squeeze() == valid_label), 3))

# Chesapeake evaluation (segmentation)

From a xid either run the evalution from script either generates the dataset embedded

In [ ]:
experiment_config = (210612708, 1)

eval_name = 'chesapeake'

_, model, train_state = trainer.load_from_xm(*experiment_config)
train_state = model.replicate_state(train_state)

config = get_config(config_name="eval_geolocation_retrieval")
eval_config = config.test_evals[eval_name]

evaluator = load.get_eval(eval_config.get('eval_name', eval_name))(
      model,
      eval_config,
  )

train, test = evaluator.get_database_representations(train_state, return_input=True)
# _, metrics, _ = evaluator.evaluate(train_state)
# print(metrics)

## PCA of dense features

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

X = train['embedding'][:,0,:]
# Fit PCA to find all components
images, pcas, labels = [], [], []
index_to_plot = [1, 8, 10, 12]
for index in index_to_plot:
  pca_all = PCA(n_components=3)

  original_image = X[index]
  pca_all.fit(original_image)
  image_shape = (16, 16)

  image_scores = pca_all.transform(original_image)

  # Normalize each channel to the 0-1 range to be a valid color
  def normalize_channel(component):
      """Scales a component vector to the 0-1 range."""
      component_min = component.min()
      component_max = component.max()
      # Avoid division by zero if the component has no variation
      if component_max == component_min:
          return np.zeros_like(component)
      return (component - component_min) / (component_max - component_min)

  r_norm = normalize_channel(image_scores[:, 0]).reshape(image_shape)
  g_norm = normalize_channel(image_scores[:, 1]).reshape(image_shape)
  b_norm = normalize_channel(image_scores[:, 2]).reshape(image_shape)

  # Stack the normalized channels into an RGB image
  rgb_components_image = np.stack([r_norm, g_norm, b_norm], axis=-1)

  # --- RESIZE THE IMAGE USING PILLOW ---
  # Convert the numpy array (0-1 float) to a Pillow Image (0-255 uint8)
  rgb_components_image_uint8 = (rgb_components_image * 255).astype(np.uint8)
  pil_image = Image.fromarray(rgb_components_image_uint8)

  # Resize the image to 256x26 using a high-quality resampling filter
  resized_pil_image = pil_image.resize((256, 256), Image.Resampling.LANCZOS)

  # Convert the resized Pillow image back to a numpy array for Matplotlib
  resized_rgb_array = np.array(resized_pil_image)

  labels.append(train['label'][index])
  pcas.append(resized_rgb_array)
  images.append(train['input'][index])

# Plot the resulting resized RGB image
fig_rgb, ax_rgb = plt.subplots(3, len(index_to_plot), figsize=(5 * len(index_to_plot), 15))
for i in range(len(index_to_plot)):
  ax_rgb[0][i].imshow(images[i])
  ax_rgb[0][i].set_title('Image')
  ax_rgb[0][i].set_xticks([])
  ax_rgb[0][i].set_yticks([])
  ax_rgb[1][i].imshow(pcas[i])
  ax_rgb[1][i].set_title('PCA')
  ax_rgb[1][i].set_xticks([])
  ax_rgb[1][i].set_yticks([])
  ax_rgb[2][i].imshow(labels[i])
  ax_rgb[2][i].set_title('Labels')
  ax_rgb[2][i].set_xticks([])
  ax_rgb[2][i].set_yticks([])
plt.show()

## Evaluation

In [ ]:
train_db = np.reshape(train['embedding'], [train['embedding'].shape[0], train['embedding'].shape[1], int(train['embedding'].shape[2] ** 0.5),
                                          int(train['embedding'].shape[2] ** 0.5), -1])
train_labels = train['label']
test_db = np.reshape(test['embedding'], [test['embedding'].shape[0], train['embedding'].shape[1], int(train['embedding'].shape[2] ** 0.5),
                                          int(train['embedding'].shape[2] ** 0.5), -1])
test_labels = test['label']
del train, test

In [ ]:
import time
from functools import partial
from typing import Any, List, Tuple

import flax
import jax
import jax.numpy as jnp
import optax
from flax.training import train_state

# --- Full Model Definition ---
import flax.linen as nn
from typing import Sequence, Optional, Literal, Callable

Activation = Callable[[jnp.ndarray], jnp.ndarray]
Shape = Tuple[int, ...]
Initializer = Callable[[jnp.ndarray, Shape, jnp.dtype], jnp.ndarray]

class ConvModule(nn.Module):
    features: int
    kernel_size: Tuple[int, int]
    stride: Tuple[int, int] = (1, 1)
    padding: str | int | Sequence[Tuple[int, int]] = 'SAME'
    dilation: Tuple[int, int] = (1, 1)
    groups: int = 1
    use_bias: bool = True
    use_norm: bool = True
    activation: Optional[Activation] = nn.relu
    order: Tuple[str, str, str] = ('conv', 'norm', 'act')
    kernel_init: Initializer = nn.initializers.kaiming_normal()
    bias_init: Initializer = nn.initializers.zeros

    @nn.compact
    def __call__(self, x: jnp.ndarray, train: bool, activate: bool = True, norm: bool = True) -> jnp.ndarray:
        in_features = x.shape[-1]
        if self.order.index('norm') > self.order.index('conv'):
            norm_features = self.features
        else:
            norm_features = in_features

        conv_layer = nn.Conv(features=self.features, kernel_size=self.kernel_size, strides=self.stride, padding=self.padding, kernel_dilation=self.dilation, feature_group_count=self.groups, use_bias=self.use_bias, kernel_init=self.kernel_init, bias_init=self.bias_init)
        norm_layer = nn.LayerNorm if self.use_norm else None

        for layer_type in self.order:
            if layer_type == 'conv':
                x = conv_layer(x)
            elif layer_type == 'norm' and norm and self.use_norm:
                assert norm_layer is not None
                x = norm_layer()(x)
            elif layer_type == 'act' and activate and self.activation is not None:
                x = self.activation(x)
        return x

class UpConvHead(nn.Module):
    features: int
    n_output_channels: int
    n_hidden_channels: int = 32

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        x = nn.Conv(features=self.features // 2, kernel_size=(3, 3), padding=1)(x)
        new_shape = (x.shape[0], x.shape[1] * 2, x.shape[2] * 2, x.shape[3])
        x = jax.image.resize(x, shape=new_shape, method='bilinear')
        x = nn.Conv(features=self.n_hidden_channels, kernel_size=(3, 3), padding=1)(x)
        x = nn.relu(x)
        x = nn.Conv(features=self.n_output_channels, kernel_size=(1, 1))(x)
        return x

class ReassembleBlocks(nn.Module):
    """
    Rearranges ViT patch features into a feature pyramid.
    **This version does NOT use a cls_token.**
    """
    in_channels: List[int]
    out_channels: List[int]
    use_batchnorm: bool = False

    def setup(self):
        self.projects = [ConvModule(features=out_ch, kernel_size=(1, 1), activation=None, use_norm=False) for _, out_ch in zip(self.in_channels, self.out_channels)]
        self.resize_layers = [
            nn.ConvTranspose(features=self.out_channels[0], kernel_size=(4, 4), strides=(4, 4)),
            nn.ConvTranspose(features=self.out_channels[1], kernel_size=(2, 2), strides=(2, 2)),
            lambda x: x,
            nn.Conv(features=self.out_channels[3], kernel_size=(3, 3), strides=(2, 2), padding=1),
        ]
        if self.use_batchnorm:
            #  self.batchnorm_layers = [nn.BatchNorm(use_running_average=True) for _ in self.in_channels]
            self.batchnorm_layers = [nn.LayerNorm() for _ in self.in_channels]


    def __call__(self, inputs: List[jnp.ndarray], train: bool) -> List[jnp.ndarray]:
        out = []
        for i, x in enumerate(inputs):
            if self.use_batchnorm:
                # Note: `BatchNorm` state is implicitly handled by Flax's mutable collections
                bn = self.batchnorm_layers[i]
                x = bn(x)
            x = self.projects[i](x, train=train)
            x = self.resize_layers[i](x)
            out.append(x)
        return out

class PreActResidualConvUnit(nn.Module):
    features: int
    @nn.compact
    def __call__(self, x: jnp.ndarray, train: bool) -> jnp.ndarray:
        conv_order = ('act', 'conv', 'norm')
        y = ConvModule(features=self.features, kernel_size=(3, 3), padding=1, use_bias=False, order=conv_order)(x, train=train)
        y = ConvModule(features=self.features, kernel_size=(3, 3), padding=1, use_bias=False, order=conv_order)(y, train=train)
        return x + y

class FeatureFusionBlock(nn.Module):
    features: int
    expand: bool = False
    @nn.compact
    def __call__(self, *inputs: jnp.ndarray, train: bool) -> jnp.ndarray:
        x = inputs[0]
        if len(inputs) == 2:
            res = inputs[1]
            if x.shape != res.shape:
                res = jax.image.resize(res, x.shape, method='bilinear')
            res_processed = PreActResidualConvUnit(features=self.features)(res, train=train)
            x = x + res_processed
        x = PreActResidualConvUnit(features=self.features)(x, train=train)
        new_shape = (x.shape[0], x.shape[1] * 2, x.shape[2] * 2, x.shape[3])
        x = jax.image.resize(x, shape=new_shape, method='bilinear')
        out_features = self.features // 2 if self.expand else self.features
        x = ConvModule(features=out_features, kernel_size=(1, 1), activation=None, use_norm=False)(x, train=train)
        return x

class DPTHead(nn.Module):
    in_channels: List[int] = (768, 768, 768, 768)
    channels: int = 256
    post_process_channels: List[int] = (128, 256, 512, 768)
    n_output_channels: int = 256
    use_batchnorm: bool = False
    expand_channels: bool = False

    def setup(self):
        self.reassemble_blocks = ReassembleBlocks(in_channels=self.in_channels, out_channels=self.post_process_channels, use_batchnorm=self.use_batchnorm)
        self.convs = [ConvModule(features=self.channels, kernel_size=(3, 3), padding=1, activation=None, use_bias=False) for _ in self.post_process_channels]
        self.fusion_blocks = [FeatureFusionBlock(features=self.channels) for _ in self.post_process_channels]
        self.project = ConvModule(features=self.channels, kernel_size=(3, 3), padding=1)
        self.conv_depth = UpConvHead(features=self.channels, n_output_channels=self.n_output_channels)

    def __call__(self, inputs: List[jnp.ndarray], train: bool) -> jnp.ndarray:
        x = self.reassemble_blocks(inputs, train=train)
        x = [self.convs[i](feature, train=train) for i, feature in enumerate(x)]
        out = self.fusion_blocks[0](x[-1], train=train)
        for i in range(1, len(self.fusion_blocks)):
            out = self.fusion_blocks[i](out, x[-(i + 1)], train=train)
        out = self.project(out, train=train)
        out = self.conv_depth(out)
        return out

In [ ]:
# Define a custom TrainState to also hold BatchNorm statistics
class DPTTrainState(train_state.TrainState):
    batch_stats: Any

def create_train_state(model, key, learning_rate, dummy_inputs):
    """Creates initial DPTTrainState with a learning rate scheduler."""
    variables = model.init(key, dummy_inputs, train=False)
    params = variables['params']
    batch_stats = variables.get('batch_stats', {})

    NUM_EPOCHS = 40
    STEPS_PER_EPOCH = train_db.shape[0] // BATCH_SIZE
    WARMUP_EPOCHS = 1

    warmup_steps = WARMUP_EPOCHS * STEPS_PER_EPOCH
    total_steps = NUM_EPOCHS * STEPS_PER_EPOCH

    lr_schedule = optax.join_schedules(
        schedules=[
            optax.linear_schedule(init_value=0.0, end_value=learning_rate, transition_steps=warmup_steps),
            optax.cosine_decay_schedule(init_value=learning_rate, decay_steps=total_steps - warmup_steps)
        ],
        boundaries=[warmup_steps]
    )

    tx = optax.adamw(learning_rate=lr_schedule)

    return DPTTrainState.create(
        apply_fn=model.apply,
        params=params,
        tx=tx,
        batch_stats=batch_stats,
    )

@partial(jax.pmap, axis_name='batch')
def train_step(state, batch, dropout_key):
    """Train for a single step, parallelized across devices."""
    inputs, targets = batch
    dropout_train_key = jax.random.fold_in(dropout_key, state.step)

    def loss_fn(params):
        logits, new_model_state = state.apply_fn(
            {'params': params, 'batch_stats': state.batch_stats},
            inputs,
            train=True,
            mutable=['batch_stats'],
            rngs={'dropout': dropout_train_key}
        )
        loss = optax.softmax_cross_entropy_with_integer_labels(logits, targets).mean()
        return loss, new_model_state

    grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
    (loss, new_model_state), grads = grad_fn(state.params)

    # Average gradients and loss across all devices
    grads = jax.lax.pmean(grads, axis_name='batch')
    loss = jax.lax.pmean(loss, axis_name='batch')

    # Update state
    new_state = state.apply_gradients(
        grads=grads,
        batch_stats=new_model_state['batch_stats']
    )
    return new_state, loss

@partial(jax.pmap, axis_name='batch')
def eval_step(state, batch):
    """Evaluate for a single step, parallelized across devices."""
    inputs, targets = batch
    logits = state.apply_fn(
        {'params': state.params, 'batch_stats': state.batch_stats},
        inputs,
        train=False
    )
    loss = optax.softmax_cross_entropy_with_integer_labels(logits, targets).mean()
    loss = jax.lax.pmean(loss, axis_name='batch')
    return loss

def shard_dict_batch(batch):
    """
    Shards a dictionary batch of data across all available devices.
    """
    num_devices = jax.local_device_count()

    # The 'image' field is a list of feature maps. We shard each one.
    sharded_images = []
    for feature_map in batch['image']:
        # Reshape to (num_devices, per_device_batch_size, H, W, C)
        sharded_fm = feature_map.reshape((num_devices, -1) + feature_map.shape[1:])
        sharded_images.append(sharded_fm)

    # The 'label' field is a single tensor. We shard it directly.
    sharded_labels = batch['label'].reshape((num_devices, -1) + batch['label'].shape[1:])

    # The pmapped train_step expects a tuple of (inputs, targets)
    return (sharded_images, sharded_labels)

In [ ]:
# --- Configuration ---
LEARNING_RATE = 1e-3
BATCH_SIZE = 256
NUM_EPOCHS = 40
PATCH_GRID_SIZE = 16

# Model architecture parameters
VIT_CHANNELS = [768, 768, 768, 768]
POST_PROCESS_CHANNELS = [128, 256, 512, 768]
FINAL_OUT_CHANNELS = 7

# --- Initialization ---
num_devices = jax.local_device_count()

train_db_split = [train_db[:, i, ...] for i in range(train_db.shape[1])]
test_db_split = [test_db[:, i, ...] for i in range(test_db.shape[1])]
# train_db_split = [normalized_train_db[:, i, ...] for i in range(train_db.shape[1])][::-1]
# test_db_split = [normalized_test_db[:, i, ...] for i in range(test_db.shape[1])][::-1]

train_slicing_components = (*train_db_split, train_labels)
train_ds = tf.data.Dataset.from_tensor_slices(train_slicing_components)

# The pipeline now ONLY shuffles and batches. It will yield a TUPLE of batched tensors.
train_ds = train_ds.shuffle(1000).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

# Create the test dataset using the same simplified logic
test_slicing_components = (*test_db_split, test_labels)
test_ds_full = tf.data.Dataset.from_tensor_slices(test_slicing_components)
test_ds_full = test_ds_full.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_iter = tfds.as_numpy(train_ds)

# --- Initialize Model and Replicate State ---
key = jax.random.PRNGKey(0)
# Split the main key
key, model_key, dropout_init_key = jax.random.split(key, 3)

model = DPTHead(
    in_channels=VIT_CHANNELS,
    channels=256,
    use_batchnorm=True,
    post_process_channels=POST_PROCESS_CHANNELS,
    n_output_channels=FINAL_OUT_CHANNELS,
)

init_dummy_inputs = [pf[0:1] for pf in train_db_split]
state = create_train_state(model, model_key, LEARNING_RATE, init_dummy_inputs)
replicated_state = flax.jax_utils.replicate(state)

# NEW: Create a unique dropout key for each GPU
dropout_keys = jax.random.split(dropout_init_key, jax.local_device_count())

In [ ]:
@jax.jit
def predict_step(state, batch):
    """Generates predictions for a single batch on one device."""
    inputs, targets = batch
    logits = state.apply_fn(
        {'params': state.params, 'batch_stats': state.batch_stats},
        inputs,
        train=False
    )
    # Get predicted class by finding the index with the highest score
    predictions = jnp.argmax(logits, axis=-1)

    # Ensure labels have the correct shape and type
    if targets.shape[-1] == 1:
        targets = jnp.squeeze(targets, axis=-1)
    targets = targets.astype(jnp.int32)

    return predictions, targets


# Convert the tf.data.Dataset to a NumPy iterator
train_iter = tfds.as_numpy(train_ds)

ious = []
print(f"Starting training for {NUM_EPOCHS} epochs...")
for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start_time = time.time()

    train_loss = 0.
    for batch_tuple in train_iter:
        # `batch_tuple` is now a simple TUPLE: (map1, map2, map3, map4, labels)

        # --- Explicitly Structure and Shard the Batch ---
        num_devices = jax.local_device_count()

        # 1. Structure the data: separate maps from labels
        input_maps = list(batch_tuple[:-1])
        labels = batch_tuple[-1]

        # 2. Shard the structured data
        sharded_images = [
            fm.reshape((num_devices, -1) + fm.shape[1:]) for fm in input_maps
        ]
        sharded_labels = labels.reshape((num_devices, -1) + labels.shape[1:])

        # This is the final (inputs, targets) tuple with the correct structure and shapes
        sharded_batch = (sharded_images, sharded_labels)

        # --- Run the Training Step ---
        replicated_state, loss = train_step(replicated_state, sharded_batch, dropout_keys)
        train_loss += loss[0]

    num_train_steps = train_db.shape[0] // BATCH_SIZE
    train_loss /= num_train_steps
    epoch_time = time.time() - epoch_start_time

    eval_start_time = time.time()
    final_state = flax.jax_utils.unreplicate(replicated_state)

    # 2. Create the iterator for the full test set
    test_iter_full = tfds.as_numpy(test_ds_full)

    # 3. Instantiate the Keras metric
    # Make sure NUM_CLASSES is defined in your configuration
    iou_metric = tf.keras.metrics.MeanIoU(num_classes=7, name='mean_iou')

    # 4. Loop through the entire dataset to calculate loss and update the metric
    eval_losses = []
    for batch_tuple in test_iter_full:
        # Restructure the tuple for the single-GPU steps
        inputs = list(batch_tuple[:-1])
        targets = batch_tuple[-1]
        batch = (inputs, targets)

        # Get predictions and update the IoU metric
        predictions, true_labels = predict_step(final_state, batch)
        iou_metric.update_state(true_labels, predictions)

    final_iou = iou_metric.result().numpy()
    eval_time = time.time() - eval_start_time
    ious.append(final_iou)
    print(f"Epoch {epoch}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Time: {epoch_time:.2f}s | mIoU: {final_iou:.4f} | Eval Time: {eval_time:.2f}s ")

In [ ]:
plt.plot(ious), np.array(ious).max()

## Silgip eval

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import functools

# Original imports
from unigeoclip.scenic.models import siglip
from unigeoclip.scenic import datasets

# --- 1. Model and Config Setup (Mostly Unchanged) ---

model_name = "SigLIP2 So400m/16 256"
model_name = "SigLIP2 B/16 256"
variant = "So400m" if model_name == "SigLIP2 So400m/16 256" else "B"

depths = (20, 13, 6) if model_name == "SigLIP2 So400m/16 256" else (9, 6, 3)

print("Initializing model...")
model = siglip.PreTrainedSigLip2Model(
    variant=variant,
    pool_type='map',
    model_name=model_name,
)

# Dummy init remains the same
dummy_inputs = np.zeros((1, 256, 256, 3), np.float32)
variables = model.init(jax.random.PRNGKey(0), dummy_inputs)

print("Loading pretrained parameters...")
loaded_model_params = model.load_pretrained_params(model_name)
params = {'params': loaded_model_params}

config = get_config()
eval_name = 'chesapeake'
eval_config = config.test_evals[eval_name]

# --- 2. Configure for Parallel TPU Execution ---

num_devices = jax.device_count()
if num_devices == 0:
    raise ValueError("No JAX devices found. This optimization requires TPUs.")

# Assume the config's batch_size is the GLOBAL batch size.
# It must be divisible by the number of devices.
global_batch_size = 64
if global_batch_size % num_devices != 0:
    print(
        f"Warning: Global batch size ({global_batch_size}) not divisible "
        f"by device count ({num_devices}). Adjusting global batch size."
    )
    # Adjust batch size to be a multiple of num_devices
    global_batch_size = (global_batch_size // num_devices) * num_devices
    if global_batch_size == 0:
         # Fallback if original batch_size < num_devices
         global_batch_size = num_devices
         print(f"Setting global batch size to {global_batch_size}.")

eval_config.batch_size_eval = global_batch_size
eval_config.batch_size = global_batch_size

print(f"Running on {num_devices} devices.")
print(f"Global batch size: {global_batch_size}")
print(f"Per-device batch size: {global_batch_size // num_devices}")

# Build the dataset. The dataset pipeline should automatically handle
# sharding the global_batch_size across devices.
dataset = datasets.build_dataset(
    eval_config.data,
    eval_config.batch_size,       # train_batch_size (global)
    eval_config.batch_size_eval,  # eval_batch_size (global)
    0
)

# --- 3. Define and Parallelize the Inference Step ---

def inference_step(params, batch):
  """
  A single inference step for one device's shard of data.
  This function will be JIT-compiled and run in parallel.
  """
  inputs = batch['input']

  # Model application
  embeddings = model.apply(
        params,
        inputs,
        get_depths=depths,
        method=model.encode_dense)

  return embeddings, batch['label']

# Use jax.pmap to create a parallel version of the inference step.
# This compiles the function and distributes it to all TPU cores.
p_inference_step = jax.pmap(inference_step, axis_name='batch')

# --- 4. Replicate Parameters to All Devices ---

print("Replicating model parameters to all devices...")
# This copies the model weights to each TPU core's memory.
p_params = jax.device_put_replicated(params, jax.local_devices())
print("Replication complete.")

import jax.tree # <--- Add this import (or jax.tree_util)

# --- 5. Refactored Feature Extraction Loop (FIXED API CALL) ---

def extract_features(iterator, p_inference_fn):
    """
    Runs feature extraction over a dataset iterator using the
    parallelized inference function.

    This version handles padding for the final incomplete batch
    and uses the correct jax.tree.map API.
    """
    labels, embed = [], []
    print("Starting feature extraction... (First batch will trigger JIT compilation)")

    for i, batch in enumerate(iterator):
        # batch is a GLOBAL batch from the iterator (e.g., shape (128, ...) or (94, ...))

        # --- 1. Handle Incomplete Batches (Padding) ---
        actual_size = batch['input'].shape[0]

        if actual_size < global_batch_size:

            # Create a new padded_batch dict
            padded_batch = {}
            for key, val in batch.items():
                # Create a correctly-sized array of zeros
                padded_val = np.zeros((global_batch_size, *val.shape[1:]), dtype=val.dtype)
                # Copy the real data into the start of the padded array
                padded_val[:actual_size] = val
                padded_batch[key] = padded_val

            batch = padded_batch # Use the padded batch for inference

        # --- 2. Manually Shard the Batch ---
        # Reshape from (global_bs, ...) to (num_devices, per_device_bs, ...)
        # e.g., (128, 256, 256, 3) -> (2, 64, 256, 256, 3)

        # *** FIX IS HERE ***
        sharded_batch = jax.tree.map(  # <--- Changed from jax.tree_map
            lambda x: x.reshape((num_devices, global_batch_size // num_devices, *x.shape[1:])),
            batch
        )

        # --- 3. Run Inference ---
        # p_inference_fn expects sharded inputs (axis 0 for both params and batch)
        batch_embeds, batch_labels = p_inference_fn(p_params, sharded_batch)

        # --- 4. Gather and Unpad ---
        # Gather results from all devices back to the host CPU
        batch_embeds = jax.device_get(batch_embeds) # Shape (num_devices, per_device_bs, dim)
        batch_labels = jax.device_get(batch_labels) # Shape (num_devices, per_device_bs, ...)

        batch_labels = batch_labels.reshape(-1, *batch_labels.shape[2:])
        batch_embeds = batch_embeds.reshape(-1, *batch_embeds.shape[2:])

        # Slice to remove padding *if* we padded this batch
        if actual_size < global_batch_size:
            batch_labels = batch_labels[:actual_size]
            batch_embeds = batch_embeds[:actual_size]

        labels.append(batch_labels)
        embed.append(batch_embeds)

        if (i + 1) % 50 == 0:
            print(f"  ... processed {i + 1} batches.")

    print("Feature extraction complete.")

    if not labels:
        print("Warning: No data found in iterator.")
        return np.array([]), np.array([])

    # Concatenate all batches into two large numpy arrays
    return np.concatenate(labels), np.concatenate(embed)

# --- 6. Run Extraction ---

print("\n--- Extracting Training Set ---")
train_label, train_embed = extract_features(dataset.train_iter(), p_inference_step)
print(f"Training set extracted: embeds {train_embed.shape}, labels {train_label.shape}")

train_db = np.reshape(train_embed, [train_embed.shape[0], train_embed.shape[1], int(train_embed.shape[2] ** 0.5),
                                          int(train_embed.shape[2] ** 0.5), -1])
train_labels = train_label

In [ ]:
print("\n--- Extracting Validation Set ---")
valid_label, valid_embed = extract_features(dataset.valid_iter(), p_inference_step)
print(f"Validation set extracted: embeds {valid_embed.shape}, labels {valid_label.shape}")

test_db = np.reshape(valid_embed, [valid_embed.shape[0], valid_embed.shape[1], int(valid_embed.shape[2] ** 0.5),
                                          int(valid_embed.shape[2] ** 0.5), -1])
test_labels = valid_label

del valid_embed, valid_label

In [ ]:
train_db = np.reshape(train_embed, [train_embed.shape[0], train_embed.shape[1], int(train_embed.shape[2] ** 0.5),
                                          int(train_embed.shape[2] ** 0.5), -1])
train_labels = train_label
del train_embed, train_label
test_db = np.reshape(valid_embed, [valid_embed.shape[0], valid_embed.shape[1], int(valid_embed.shape[2] ** 0.5),
                                          int(valid_embed.shape[2] ** 0.5), -1])
test_labels = valid_label
print(train_db.shape, test_db.shape)

In [ ]:
import time
from functools import partial
from typing import Any, List, Tuple

import flax
import jax
import jax.numpy as jnp
import optax
from flax.training import train_state

# --- Full Model Definition ---
import flax.linen as nn
from typing import Sequence, Optional, Literal, Callable

Activation = Callable[[jnp.ndarray], jnp.ndarray]
Shape = Tuple[int, ...]
Initializer = Callable[[jnp.ndarray, Shape, jnp.dtype], jnp.ndarray]

class ConvModule(nn.Module):
    features: int
    kernel_size: Tuple[int, int]
    stride: Tuple[int, int] = (1, 1)
    padding: str | int | Sequence[Tuple[int, int]] = 'SAME'
    dilation: Tuple[int, int] = (1, 1)
    groups: int = 1
    use_bias: bool = True
    use_norm: bool = True
    activation: Optional[Activation] = nn.relu
    order: Tuple[str, str, str] = ('conv', 'norm', 'act')
    kernel_init: Initializer = nn.initializers.kaiming_normal()
    bias_init: Initializer = nn.initializers.zeros

    @nn.compact
    def __call__(self, x: jnp.ndarray, train: bool, activate: bool = True, norm: bool = True) -> jnp.ndarray:
        in_features = x.shape[-1]
        if self.order.index('norm') > self.order.index('conv'):
            norm_features = self.features
        else:
            norm_features = in_features

        conv_layer = nn.Conv(features=self.features, kernel_size=self.kernel_size, strides=self.stride, padding=self.padding, kernel_dilation=self.dilation, feature_group_count=self.groups, use_bias=self.use_bias, kernel_init=self.kernel_init, bias_init=self.bias_init)
        norm_layer = nn.LayerNorm if self.use_norm else None

        for layer_type in self.order:
            if layer_type == 'conv':
                x = conv_layer(x)
            elif layer_type == 'norm' and norm and self.use_norm:
                assert norm_layer is not None
                x = norm_layer()(x)
            elif layer_type == 'act' and activate and self.activation is not None:
                x = self.activation(x)
        return x

class UpConvHead(nn.Module):
    features: int
    n_output_channels: int
    n_hidden_channels: int = 32

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        x = nn.Conv(features=self.features // 2, kernel_size=(3, 3), padding=1)(x)
        new_shape = (x.shape[0], x.shape[1] * 2, x.shape[2] * 2, x.shape[3])
        x = jax.image.resize(x, shape=new_shape, method='bilinear')
        x = nn.Conv(features=self.n_hidden_channels, kernel_size=(3, 3), padding=1)(x)
        x = nn.relu(x)
        x = nn.Conv(features=self.n_output_channels, kernel_size=(1, 1))(x)
        return x

class ReassembleBlocks(nn.Module):
    """
    Rearranges ViT patch features into a feature pyramid.
    **This version does NOT use a cls_token.**
    """
    in_channels: List[int]
    out_channels: List[int]
    use_batchnorm: bool = False

    def setup(self):
        self.projects = [ConvModule(features=out_ch, kernel_size=(1, 1), activation=None, use_norm=False) for _, out_ch in zip(self.in_channels, self.out_channels)]
        self.resize_layers = [
            nn.ConvTranspose(features=self.out_channels[0], kernel_size=(4, 4), strides=(4, 4)),
            nn.ConvTranspose(features=self.out_channels[1], kernel_size=(2, 2), strides=(2, 2)),
            lambda x: x,
            nn.Conv(features=self.out_channels[3], kernel_size=(3, 3), strides=(2, 2), padding=1),
        ]
        if self.use_batchnorm:
            #  self.batchnorm_layers = [nn.BatchNorm(use_running_average=True) for _ in self.in_channels]
            self.batchnorm_layers = [nn.LayerNorm() for _ in self.in_channels]


    def __call__(self, inputs: List[jnp.ndarray], train: bool) -> List[jnp.ndarray]:
        out = []
        for i, x in enumerate(inputs):
            if self.use_batchnorm:
                # Note: `BatchNorm` state is implicitly handled by Flax's mutable collections
                bn = self.batchnorm_layers[i]
                x = bn(x)
            x = self.projects[i](x, train=train)
            x = self.resize_layers[i](x)
            out.append(x)
        return out

class PreActResidualConvUnit(nn.Module):
    features: int
    @nn.compact
    def __call__(self, x: jnp.ndarray, train: bool) -> jnp.ndarray:
        conv_order = ('act', 'conv', 'norm')
        y = ConvModule(features=self.features, kernel_size=(3, 3), padding=1, use_bias=False, order=conv_order)(x, train=train)
        y = ConvModule(features=self.features, kernel_size=(3, 3), padding=1, use_bias=False, order=conv_order)(y, train=train)
        return x + y

class FeatureFusionBlock(nn.Module):
    features: int
    expand: bool = False
    @nn.compact
    def __call__(self, *inputs: jnp.ndarray, train: bool) -> jnp.ndarray:
        x = inputs[0]
        if len(inputs) == 2:
            res = inputs[1]
            if x.shape != res.shape:
                res = jax.image.resize(res, x.shape, method='bilinear')
            res_processed = PreActResidualConvUnit(features=self.features)(res, train=train)
            x = x + res_processed
        x = PreActResidualConvUnit(features=self.features)(x, train=train)
        new_shape = (x.shape[0], x.shape[1] * 2, x.shape[2] * 2, x.shape[3])
        x = jax.image.resize(x, shape=new_shape, method='bilinear')
        out_features = self.features // 2 if self.expand else self.features
        x = ConvModule(features=out_features, kernel_size=(1, 1), activation=None, use_norm=False)(x, train=train)
        return x

class DPTHead(nn.Module):
    in_channels: List[int] = (768, 768, 768, 768)
    channels: int = 256
    post_process_channels: List[int] = (128, 256, 512, 768)
    n_output_channels: int = 256
    use_batchnorm: bool = False
    expand_channels: bool = False

    def setup(self):
        self.reassemble_blocks = ReassembleBlocks(in_channels=self.in_channels, out_channels=self.post_process_channels, use_batchnorm=self.use_batchnorm)
        self.convs = [ConvModule(features=self.channels, kernel_size=(3, 3), padding=1, activation=None, use_bias=False) for _ in self.post_process_channels]
        self.fusion_blocks = [FeatureFusionBlock(features=self.channels) for _ in self.post_process_channels]
        self.project = ConvModule(features=self.channels, kernel_size=(3, 3), padding=1)
        self.conv_depth = UpConvHead(features=self.channels, n_output_channels=self.n_output_channels)

    def __call__(self, inputs: List[jnp.ndarray], train: bool) -> jnp.ndarray:
        x = self.reassemble_blocks(inputs, train=train)
        x = [self.convs[i](feature, train=train) for i, feature in enumerate(x)]
        out = self.fusion_blocks[0](x[-1], train=train)
        for i in range(1, len(self.fusion_blocks)):
            out = self.fusion_blocks[i](out, x[-(i + 1)], train=train)
        out = self.project(out, train=train)
        out = self.conv_depth(out)
        return out

In [ ]:
# Define a custom TrainState to also hold BatchNorm statistics
class DPTTrainState(train_state.TrainState):
    batch_stats: Any

def create_train_state(model, key, learning_rate, dummy_inputs):
    """Creates initial DPTTrainState with a learning rate scheduler."""
    variables = model.init(key, dummy_inputs, train=False)
    params = variables['params']
    batch_stats = variables.get('batch_stats', {})

    NUM_EPOCHS = 40
    STEPS_PER_EPOCH = train_db.shape[0] // BATCH_SIZE
    WARMUP_EPOCHS = 1

    warmup_steps = WARMUP_EPOCHS * STEPS_PER_EPOCH
    total_steps = NUM_EPOCHS * STEPS_PER_EPOCH

    lr_schedule = optax.join_schedules(
        schedules=[
            optax.linear_schedule(init_value=0.0, end_value=learning_rate, transition_steps=warmup_steps),
            optax.cosine_decay_schedule(init_value=learning_rate, decay_steps=total_steps - warmup_steps)
        ],
        boundaries=[warmup_steps]
    )

    tx = optax.adam(learning_rate=lr_schedule)

    return DPTTrainState.create(
        apply_fn=model.apply,
        params=params,
        tx=tx,
        batch_stats=batch_stats,
    )

@partial(jax.pmap, axis_name='batch')
def train_step(state, batch, dropout_key):
    """Train for a single step, parallelized across devices."""
    inputs, targets = batch
    dropout_train_key = jax.random.fold_in(dropout_key, state.step)

    def loss_fn(params):
        logits, new_model_state = state.apply_fn(
            {'params': params, 'batch_stats': state.batch_stats},
            inputs,
            train=True,
            mutable=['batch_stats'],
            rngs={'dropout': dropout_train_key}
        )
        loss = optax.softmax_cross_entropy_with_integer_labels(logits, targets).mean()
        return loss, new_model_state

    grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
    (loss, new_model_state), grads = grad_fn(state.params)

    # Average gradients and loss across all devices
    grads = jax.lax.pmean(grads, axis_name='batch')
    loss = jax.lax.pmean(loss, axis_name='batch')

    # Update state
    new_state = state.apply_gradients(
        grads=grads,
        batch_stats=new_model_state['batch_stats']
    )
    return new_state, loss

@partial(jax.pmap, axis_name='batch')
def eval_step(state, batch):
    """Evaluate for a single step, parallelized across devices."""
    inputs, targets = batch
    logits = state.apply_fn(
        {'params': state.params, 'batch_stats': state.batch_stats},
        inputs,
        train=False
    )
    loss = optax.softmax_cross_entropy_with_integer_labels(logits, targets).mean()
    loss = jax.lax.pmean(loss, axis_name='batch')
    return loss

def shard_dict_batch(batch):
    """
    Shards a dictionary batch of data across all available devices.
    """
    num_devices = jax.local_device_count()

    # The 'image' field is a list of feature maps. We shard each one.
    sharded_images = []
    for feature_map in batch['image']:
        # Reshape to (num_devices, per_device_batch_size, H, W, C)
        sharded_fm = feature_map.reshape((num_devices, -1) + feature_map.shape[1:])
        sharded_images.append(sharded_fm)

    # The 'label' field is a single tensor. We shard it directly.
    sharded_labels = batch['label'].reshape((num_devices, -1) + batch['label'].shape[1:])

    # The pmapped train_step expects a tuple of (inputs, targets)
    return (sharded_images, sharded_labels)

In [ ]:
# --- Configuration ---
LEARNING_RATE = 1e-3
BATCH_SIZE = 128
NUM_EPOCHS = 40
PATCH_GRID_SIZE = 16

# Model architecture parameters
VIT_CHANNELS = [1152, 1152, 1152, 1152]
POST_PROCESS_CHANNELS = [256, 512, 768, 1152]
FINAL_OUT_CHANNELS = 7

# --- Initialization ---
num_devices = jax.local_device_count()

train_db_split = [train_db[:, i, ...] for i in range(train_db.shape[1])]
test_db_split = [test_db[:, i, ...] for i in range(test_db.shape[1])]
# train_db_split = [normalized_train_db[:, i, ...] for i in range(train_db.shape[1])][::-1]
# test_db_split = [normalized_test_db[:, i, ...] for i in range(test_db.shape[1])][::-1]

train_slicing_components = (*train_db_split, train_labels)
train_ds = tf.data.Dataset.from_tensor_slices(train_slicing_components)

# The pipeline now ONLY shuffles and batches. It will yield a TUPLE of batched tensors.
train_ds = train_ds.shuffle(1000).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

# Create the test dataset using the same simplified logic
test_slicing_components = (*test_db_split, test_labels)
test_ds_full = tf.data.Dataset.from_tensor_slices(test_slicing_components)
test_ds_full = test_ds_full.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_iter = tfds.as_numpy(train_ds)

# --- Initialize Model and Replicate State ---
key = jax.random.PRNGKey(0)
# Split the main key
key, model_key, dropout_init_key = jax.random.split(key, 3)

model = DPTHead(
    in_channels=VIT_CHANNELS,
    channels=256,
    use_batchnorm=True,
    post_process_channels=POST_PROCESS_CHANNELS,
    n_output_channels=FINAL_OUT_CHANNELS,
)

init_dummy_inputs = [pf[0:1] for pf in train_db_split]
state = create_train_state(model, model_key, LEARNING_RATE, init_dummy_inputs)
replicated_state = flax.jax_utils.replicate(state)

# NEW: Create a unique dropout key for each GPU
dropout_keys = jax.random.split(dropout_init_key, jax.local_device_count())

In [ ]:
@jax.jit
def predict_step(state, batch):
    """Generates predictions for a single batch on one device."""
    inputs, targets = batch
    logits = state.apply_fn(
        {'params': state.params, 'batch_stats': state.batch_stats},
        inputs,
        train=False
    )
    # Get predicted class by finding the index with the highest score
    predictions = jnp.argmax(logits, axis=-1)

    # Ensure labels have the correct shape and type
    if targets.shape[-1] == 1:
        targets = jnp.squeeze(targets, axis=-1)
    targets = targets.astype(jnp.int32)

    return predictions, targets


# Convert the tf.data.Dataset to a NumPy iterator
train_iter = tfds.as_numpy(train_ds)

ious = []
print(f"Starting training for {NUM_EPOCHS} epochs...")
for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start_time = time.time()

    train_loss = 0.
    for batch_tuple in train_iter:
        # `batch_tuple` is now a simple TUPLE: (map1, map2, map3, map4, labels)

        # --- Explicitly Structure and Shard the Batch ---
        num_devices = jax.local_device_count()

        # 1. Structure the data: separate maps from labels
        input_maps = list(batch_tuple[:-1])
        labels = batch_tuple[-1]

        # 2. Shard the structured data
        sharded_images = [
            fm.reshape((num_devices, -1) + fm.shape[1:]) for fm in input_maps
        ]
        sharded_labels = labels.reshape((num_devices, -1) + labels.shape[1:])

        # This is the final (inputs, targets) tuple with the correct structure and shapes
        sharded_batch = (sharded_images, sharded_labels)

        # --- Run the Training Step ---
        replicated_state, loss = train_step(replicated_state, sharded_batch, dropout_keys)
        train_loss += loss[0]

    num_train_steps = train_db.shape[0] // BATCH_SIZE
    train_loss /= num_train_steps
    epoch_time = time.time() - epoch_start_time

    eval_start_time = time.time()
    final_state = flax.jax_utils.unreplicate(replicated_state)

    # 2. Create the iterator for the full test set
    test_iter_full = tfds.as_numpy(test_ds_full)

    # 3. Instantiate the Keras metric
    # Make sure NUM_CLASSES is defined in your configuration
    iou_metric = tf.keras.metrics.MeanIoU(num_classes=7, name='mean_iou')

    # 4. Loop through the entire dataset to calculate loss and update the metric
    eval_losses = []
    for batch_tuple in test_iter_full:
        # Restructure the tuple for the single-GPU steps
        inputs = list(batch_tuple[:-1])
        targets = batch_tuple[-1]
        batch = (inputs, targets)

        # Get predictions and update the IoU metric
        predictions, true_labels = predict_step(final_state, batch)
        iou_metric.update_state(true_labels, predictions)

    final_iou = iou_metric.result().numpy()
    eval_time = time.time() - eval_start_time
    ious.append(final_iou)
    print(f"Epoch {epoch}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Time: {epoch_time:.2f}s | mIoU: {final_iou:.4f} | Eval Time: {eval_time:.2f}s ")

In [ ]:
plt.plot(ious)

# MDAS evaluation

From a xid either run the evalution from script either generates the dataset embedded

In [ ]:
experiment_config = (211109853, 1)

eval_name = 'mdas_dsm'

_, model, train_state = trainer.load_from_xm(*experiment_config)
train_state = model.replicate_state(train_state)

config = get_config(config_name="eval_geolocation_retrieval")
eval_config = config.test_evals[eval_name]

evaluator = load.get_eval(eval_config.get('eval_name', eval_name))(
    model,
    eval_config,
)

# _, metrics, _ = evaluator.evaluate(train_state)
# print(metrics)
train, test = evaluator.get_database_representations(train_state)
print(train['label'].shape, test['label'].shape)

In [ ]:
train_db = np.reshape(train['embedding'], [train['embedding'].shape[0], int(train['embedding'].shape[1] ** 0.5),
                                          int(train['embedding'].shape[1] ** 0.5), -1])
train_labels = train['label']
test_db = np.reshape(test['embedding'], [test['embedding'].shape[0], int(train['embedding'].shape[1] ** 0.5),
                                          int(train['embedding'].shape[1] ** 0.5), -1])
test_labels = test['label']
del train, test

In [ ]:
train_db.shape, test_db.shape, train_labels.shape, test_labels.shape

In [ ]:
import tensorflow as tf
from tensorflow.keras import optimizers
from functools import partial

def reshape_output(y_pred, num_classes, patch_size):
    """Applies the custom reshaping and transposing logic."""
    batch_size = tf.shape(y_pred)[0]
    height = tf.shape(y_pred)[1]
    width = tf.shape(y_pred)[2]

    # # Keras requires channel dimension to be known, so we get it statically
    # channels = y_pred.shape[-1]

    y_pred = tf.reshape(y_pred, (batch_size, height, width, patch_size, patch_size, num_classes))
    y_pred = tf.transpose(y_pred, perm=[0, 1, 3, 2, 4, 5])
    y_pred = tf.reshape(y_pred, (batch_size, height * patch_size, width, patch_size, num_classes))
    y_pred = tf.reshape(y_pred, (batch_size, height * patch_size, width * patch_size, num_classes))
    return y_pred

# lrs = [0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001, 0.0005, 0.0002, 0.0001]
def get_loss_fn(y_true, y_pred):
  y_pred = y_pred.reshape((-1, 13))
  loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, ignore_class=-1)
  return loss(y_true, y_pred)

class ArgmaxMetricWrapper(tf.keras.metrics.Metric):
    """A wrapper that applies argmax to predictions before updating a metric."""

    def __init__(self, metric, name='argmax_metric_wrapper', **kwargs):
        super().__init__(name=name, **kwargs)
        # The metric to wrap (e.g., an instance of MeanIoU)
        self.metric = metric

    def update_state(self, y_true, y_pred, sample_weight=None):
        # Convert model predictions (logits/probabilities) to class indices
        y_pred_argmax = tf.argmax(y_pred, axis=-1)

        # Pass the true labels and the class predictions to the wrapped metric
        self.metric.update_state(y_true, y_pred_argmax, sample_weight)

    def result(self):
        return self.metric.result()

    def reset_state(self):
        self.metric.reset_state()

class Acc(tf.keras.metrics.Metric):
    """
    A custom accuracy metric for semantic segmentation that ignores a specific class.

    Args:
      name: (Optional) string name of the metric instance.
      ignore_class_id: The class ID to ignore when calculating accuracy.
      **kwargs: Additional keyword arguments.
    """
    def __init__(self, name='masked_accuracy', ignore_class_id=-1, **kwargs):
        super(Acc, self).__init__(name=name, **kwargs)
        self.ignore_class_id = ignore_class_id

        # Use add_weight to create state variables that are tracked by the layer
        self.total_correct = self.add_weight(
            name='total_correct', initializer='zeros', dtype=tf.float32
        )
        self.total_counted = self.add_weight(
            name='total_counted', initializer='zeros', dtype=tf.float32
        )

    def update_state(self, y_true, y_pred, sample_weight=None):
        # Convert model predictions (logits/probabilities) to class labels
        y_pred_labels = tf.argmax(y_pred, axis=-1)

        # Ensure y_true and y_pred_labels have the same integer type
        y_true = tf.cast(y_true, y_pred_labels.dtype)

        # 1. Create a mask to identify pixels that should be included in the calculation
        # These are all pixels where the true label is NOT the ignored class
        mask = tf.not_equal(y_true, self.ignore_class_id)

        # 2. Find which of the valid pixels were correctly classified
        correct_predictions = tf.equal(y_true, y_pred_labels)

        # 3. Use the mask to zero out the ignored pixels from the correctness check
        masked_correct = tf.logical_and(correct_predictions, mask)

        # 4. Update the state variables
        # Increment total_correct by the number of correctly classified valid pixels
        self.total_correct.assign_add(tf.reduce_sum(tf.cast(masked_correct, tf.float32)))

        # Increment total_counted by the total number of valid pixels
        self.total_counted.assign_add(tf.reduce_sum(tf.cast(mask, tf.float32)))

    def result(self):
        # Calculate the final accuracy, avoiding division by zero
        return tf.math.divide_no_nan(self.total_correct, self.total_counted)

    def reset_state(self):
        # Reset the state of the metric at the start of each epoch
        self.total_correct.assign(0.0)
        self.total_counted.assign(0.0)


# lrs = [0.02, 0.01, 0.005, 0.002, 0.001, 0.0005, 0.0002, 0.0001]
lrs = [0.005]
for lr in lrs:
  optim = optimizers.SGD(learning_rate=lr)

  # Compile the model for binary classification
  module_list = []
  # Output layer.
  module_list.append(tf.keras.layers.Dense(13*4))
  module_list.append(tf.keras.layers.Lambda(partial(reshape_output, num_classes=13, patch_size=2)))
  model = tf.keras.Sequential(module_list)

  model.compile(
      optimizer=optim,
      loss=get_loss_fn,
      metrics=[ArgmaxMetricWrapper(
            tf.keras.metrics.MeanIoU(num_classes=13, ignore_class=-1),
            name='mIoU' ),
              Acc(
              name='acc'
               )
        # ArgmaxMetricWrapper(
        #     tf.keras.metrics.CategoricalAccuracy(),
        #     name='acc')
        ],
  )
  history = model.fit(
      train_db,
      train_labels,
      epochs=200,
      batch_size=64,
      validation_data=(test_db, test_labels),
      verbose=True,
  )
  y_pred = model.predict(test_db, verbose=False)
  predicted_classes = tf.argmax(y_pred, axis=-1)
  mean_iou = tf.keras.metrics.MeanIoU(num_classes=13, ignore_class=-1)
  mean_iou.update_state(test_labels, predicted_classes)
  acc = Acc(name='acc')
  acc.update_state(test_labels, y_pred)
  print(lr, mean_iou.result().numpy())

In [ ]:
history_dict = history.history

# print(history_dict.keys())
train_loss = history_dict['loss']
train_miou = history_dict['mIoU']
train_accuracy = history_dict['acc']
val_loss = history_dict['val_loss']
val_miou = history_dict['val_mIoU']
val_accuracy = history_dict['val_acc']

# Create a figure with two subplots
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 5))

# Plot training & validation loss values
ax1.plot(train_loss, label='Training Loss')
ax1.plot(val_loss, label='Validation Loss')
ax1.set_title('Model Loss')
ax1.set_ylabel('Loss')
ax1.set_xlabel('Epoch')
ax1.legend(loc='upper right')
ax1.grid(True)


# Plot training & validation accuracy values
ax2.plot(train_accuracy, label='Training acc')
ax2.plot(val_accuracy, label='Validation acc')
ax2.set_title('Model Accuracy')
ax2.set_ylabel('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend(loc='lower right')
ax2.grid(True)

# Plot training & validation mIoU values
ax3.plot(train_miou, label='Training mIoU')
ax3.plot(val_miou, label='Validation mIoU')
ax3.set_title('Model Accuracy')
ax3.set_ylabel('mIoU')
ax3.set_xlabel('Epoch')
ax3.legend(loc='lower right')
ax3.grid(True)

plt.show()

# Geolocation plots

In [ ]:
y_data = {
    'Ensemble': [
        [0.4563,0.4777,0.5043,0.5412,0.5796,0.6007,0.665,0.7087],
        [0.0285,0.0305,0.0333,0.0382,0.0484,0.0594,0.1281,0.1987],
        [0.2093,0.2197,0.233,0.2541,0.2869,0.3141,0.4328,0.519],
        [0.3661,0.3828,0.4027,0.4325,0.4709,0.499,0.5958,0.656],
        [0.4326,0.4511,0.4722,0.5026,0.5388,0.563,0.6427,0.694],
        [0.2558,0.2685,0.2841,0.3079,0.3423,0.3709,0.4848,0.5652],
        [0.463,0.4848,0.5114,0.5475,0.585,0.6061,0.6694,0.7118],
        [0.5623,0.5857,0.6109,0.6411,0.6689,0.684,0.732,0.7639],
        [0.5375,0.5606,0.5867,0.6185,0.649,0.6655,0.716,0.7505],
        [0.4379,0.4571,0.4797,0.5108,0.5458,0.5682,0.6439,0.6925],
        [0.3093,0.3249,0.3447,0.3745,0.4135,0.4411,0.5363,0.5986],
        [0.6699,0.6939,0.7167,0.7406,0.7604,0.7709,0.8031,0.8243],
        [0.4726,0.4932,0.5161,0.5464,0.5807,0.6031,0.6757,0.7208],
        [0.3269,0.3449,0.3696,0.4071,0.4499,0.4776,0.5701,0.6309],
        [0.5401,0.561,0.5828,0.6081,0.6341,0.6508,0.706,0.7431],
        [0.5983,0.6211,0.6429,0.6671,0.6912,0.7058,0.7523,0.7829],
        [0.425,0.4466,0.4752,0.5143,0.5558,0.5802,0.6537,0.7005],
        [0.5066,0.5287,0.5552,0.5885,0.6211,0.64,0.698,0.7365],
        [0.5221,0.545,0.5711,0.6031,0.6353,0.6537,0.7082,0.7445],
        [0.7426,0.7669,0.7669,0.8077,0.8224,0.8295,0.8513,0.8667],
        [0.5171,0.5394,0.565,0.5978,0.6294,0.6477,0.7036,0.7409],
        [0.5065,0.529,0.555,0.5885,0.6221,0.6419,0.7007,0.7389],
        [0.4845,0.5079,0.5375,0.575,0.6097,0.6291,0.6886,0.7271],
        [0.4582,0.4809,0.5105,0.5521,0.5949,0.6195,0.682,0.7217],
        [0.4621,0.4855,0.5156,0.5567,0.5994,0.6227,0.6829,0.7224],
        [0.5081,0.5311,0.557,0.5905,0.6226,0.6399,0.6934,0.7311],
        [0.4708,0.4951,0.5262,0.5656,0.6005,0.6197,0.6791,0.7191],
        [0.6009,0.6258,0.6522,0.6821,0.7085,0.7227,0.7661,0.7951],
        [0.6042,0.6291,0.6553,0.685,0.7116,0.7259,0.7693,0.7979],
        [0.6203,0.6438,0.6663,0.6905,0.7123,0.7253,0.7668,0.7937],
        [],
        [],
        [],
    ],
    'Aerial': [
        [0.0428,0.4143,0.4386,0.4738,0.517,0.5451,0.6291,0.6811],
        [0.0033,0.0307,0.0337,0.0389,0.0481,0.058,0.123,0.193],
        [0.0114,0.1114,0.1203,0.1353,0.1608,0.1849,0.3132,0.4144],
        [0.0247,0.2418,0.2568,0.2808,0.3182,0.3512,0.4815,0.5635],
        [0.031,0.3016,0.3189,0.3451,0.3829,0.4132,0.5261,0.5971],
        [0.0187,0.1786,0.1915,0.2119,0.244,0.2723,0.4003,0.4947],
        [0.0426,0.4131,0.4375,0.4724,0.515,0.5431,0.6274,0.6793],
        [0.0563,0.5321,0.5565,0.5863,0.6188,0.64,0.7048,0.7443],
        [0.053,0.5062,0.5312,0.5628,0.5975,0.6197,0.6878,0.7296],
        [0.0384,0.3722,0.3944,0.4261,0.4664,0.494,0.5883,0.6471],
        [0.0239,0.2281,0.2454,0.2719,0.3095,0.3391,0.4536,0.53],
        [0.0677,0.6398,0.6627,0.6885,0.7133,0.7282,0.7728,0.8009],
        [0.0411,0.3967,0.4201,0.4524,0.4927,0.5214,0.6132,0.6697],
        [0.031,0.3093,0.3336,0.3705,0.4166,0.4467,0.546,0.6106],
        [0.0552,0.5199,0.5423,0.5691,0.5983,0.6176,0.6822,0.7234],
        [0.0647,0.6068,0.6293,0.6549,0.6803,0.6963,0.746,0.7787],
        [0.0429,0.4234,0.4526,0.4927,0.5368,0.5623,0.638,0.687],
        [0.0514,0.4925,0.5194,0.5549,0.5917,0.6123,0.6764,0.718],
        [0.0562,0.5402,0.5668,0.5994,0.6321,0.6504,0.7055,0.7421],
        [0.0768,0.7169,0.7392,0.7628,0.7833,0.7941,0.8274,0.8479],
        [0.0523,0.4989,0.5247,0.5585,0.5946,0.6151,0.6784,0.7198],
        [0.055,0.5219,0.548,0.5815,0.6159,0.6352,0.6928,0.7311],
        [0.0479,0.4695,0.499,0.537,0.5763,0.5986,0.6648,0.7076],
        [0.0461,0.4435,0.4731,0.5146,0.5619,0.5895,0.6581,0.7025],
        [0.0495,0.4809,0.5109,0.5511,0.594,0.6184,0.6806,0.7212],
        [0.0654,0.6203,0.6468,0.6759,0.702,0.7168,0.7619,0.7915],
        [0.065,0.6188,0.6457,0.6747,0.7015,0.7166,0.762,0.7916],
        [0.067,0.6246,0.6462,0.6709,0.6947,0.7091,0.7552,0.7856],
        [0.0489,0.4656,0.4907,0.5249,0.5646,0.5886,0.6598,0.7052],
        [0.0486,0.48,0.5114,0.5517,0.5901,0.6117,0.675,0.717],
        [],
        [],
        [],
    ],
    'Latlng': [
        [0.2411,0.2594,0.2959,0.3709,0.4459,0.4795,0.5745,0.6356],
        [0.0011,0.0012,0.0014,0.0023,0.0071,0.0147,0.0654,0.1245],
        [0.0122,0.0131,0.0155,0.0224,0.0387,0.0567,0.1732,0.2827],
        [0.0863,0.0934,0.1089,0.147,0.2056,0.2475,0.4072,0.5032],
        [0.1088,0.1176,0.1358,0.1801,0.2421,0.2836,0.4355,0.5269],
        [0.0145,0.0157,0.0187,0.0271,0.0454,0.0654,0.1937,0.3074],
        [0.2612,0.2807,0.3186,0.3922,0.4632,0.494,0.5853,0.6434],
        [0.2609,0.2806,0.3179,0.3902,0.4577,0.4879,0.58,0.6396],
        [0.2761,0.2965,0.3345,0.4064,0.4717,0.5009,0.5873,0.6443],
        [0.2663,0.2846,0.3158,0.3663,0.4178,0.4496,0.5547,0.6201],
        [0.1278,0.138,0.1593,0.2046,0.2603,0.2928,0.4039,0.4744],
        [0.3052,0.3269,0.3676,0.44,0.4997,0.524,0.5997,0.6516],
        [0.2122,0.2277,0.2593,0.3224,0.3912,0.4299,0.548,0.6187],
        [0.1585,0.1694,0.1895,0.2267,0.2783,0.3142,0.4359,0.5168],
        [0.3072,0.3251,0.3498,0.3837,0.4214,0.4478,0.5423,0.6054],
        [0.4994,0.5238,0.5538,0.5885,0.6199,0.6384,0.6965,0.7357],
        [0.3999,0.4225,0.4546,0.4983,0.5412,0.5653,0.6394,0.6879],
        [0.4327,0.4566,0.4892,0.5322,0.5701,0.5914,0.6572,0.7007],
        [0.4603,0.4848,0.5178,0.56,0.595,0.6148,0.6739,0.7155],
        [0.3888,0.4142,0.4579,0.5266,0.5734,0.5926,0.6534,0.6965],
        [0.4358,0.4595,0.4918,0.535,0.5737,0.5951,0.6602,0.7043],
        [0.449,0.473,0.5055,0.5478,0.585,0.6054,0.6673,0.7092],
        [0.4101,0.4343,0.47,0.5183,0.5592,0.5808,0.6482,0.6932],
        [0.367,0.3895,0.425,0.4807,0.5348,0.5639,0.6359,0.6826],
        [0.375,0.3985,0.4363,0.4934,0.5468,0.5738,0.6416,0.6882],
        [0.3609,0.384,0.4224,0.4838,0.5354,0.5596,0.6343,0.6835],
        [0.3596,0.3829,0.4218,0.4827,0.5346,0.5588,0.6336,0.6837],
        [0.4683,0.4932,0.5261,0.5659,0.5993,0.6191,0.6803,0.721],
        [0.3156,0.3377,0.3785,0.4489,0.5082,0.5347,0.6131,0.666],
        [0.3777,0.402,0.4425,0.5011,0.5472,0.5715,0.6444,0.6917],
        [],
        [],
        [],
        [0.198,0.2138,0.2457,0.3157,0.3935,0.4268,0.5254,0.5891],
        [],
        [],
    ],
    'cell_codes': [
        [0.0018,0.0426,0.1692,0.2971,0.3794,0.4149,0.518,0.5883],
        [0,0.0001,0.0003,0.001,0.0043,0.0094,0.0481,0.0984],
        [0.0001,0.0018,0.007,0.0153,0.0299,0.0458,0.1578,0.2692],
        [0.0004,0.0093,0.0412,0.0843,0.1366,0.1745,0.3353,0.443],
        [0.0005,0.0126,0.0512,0.1012,0.1566,0.1934,0.3484,0.4536],
        [0.0001,0.002,0.0082,0.0172,0.0338,0.0515,0.1751,0.2915],
        [0.002,0.0484,0.1831,0.3121,0.3881,0.4219,0.5232,0.5906],
        [0.0018,0.0406,0.1536,0.2661,0.3373,0.372,0.4856,0.5623],
        [0.0022,0.0519,0.1954,0.3216,0.3912,0.4232,0.5221,0.5896],
        [0.0008,0.0214,0.076,0.1278,0.1788,0.217,0.36,0.4556],
        [0.001,0.0231,0.0899,0.1624,0.2238,0.259,0.3906,0.4828],
        [0.0025,0.0579,0.2151,0.3469,0.4093,0.4353,0.5222,0.5848],
        [0.0008,0.0204,0.0815,0.1508,0.2144,0.2551,0.4045,0.4983],
        [0.0002,0.0053,0.0195,0.0373,0.0657,0.0894,0.1995,0.2912],
        [0.0027,0.0559,0.1526,0.2106,0.2575,0.2915,0.4147,0.504],
        [0.0053,0.1024,0.264,0.3479,0.3997,0.4335,0.5417,0.6142],
        [0.0044,0.0862,0.2462,0.3446,0.413,0.4479,0.5492,0.6161],
        [0.0046,0.0924,0.2626,0.3616,0.4219,0.454,0.5506,0.616],
        [0.0048,0.1012,0.285,0.3866,0.4435,0.4743,0.5664,0.6284],
        [0.0036,0.0804,0.2843,0.4219,0.4727,0.4947,0.5677,0.6221],
        [0.0047,0.0937,0.2676,0.3673,0.4271,0.4591,0.5546,0.6195],
        [0.0048,0.0975,0.2758,0.3761,0.436,0.468,0.563,0.6262],
        [],
        [],
        [],
        [0.0032,0.0758,0.2481,0.3685,0.4314,0.4596,0.5497,0.6137],
        [0.0034,0.0756,0.2467,0.3677,0.431,0.4596,0.551,0.6143],
        [0.0049,0.1011,0.2796,0.3711,0.4206,0.4509,0.5477,0.6141],
        [0.0028,0.0625,0.2305,0.365,0.4306,0.4601,0.5497,0.6118],
        [0.0009,0.0205,0.0809,0.1425,0.2018,0.2442,0.3855,0.4728],
        [],
        [],
        [0.0005,0.0108,0.045,0.0882,0.126,0.1421,0.1853,0.2142],
        [0.0204,0.2577,0.3862,0.4468,0.4879,0.513,0.5923,0.6493],
        [0.0011,0.1029,0.184,0.241,0.2708,0.3763,0.4506],
    ]
}

x_values = [10, 50, 100, 200, 500, 1000, 5000, 10000]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, (graph_title, y_values) in enumerate(y_data.items()):
  graph_title = 'Retrieval with ' + graph_title + ' database'
  ax = axes[i]
  # ax.plot(x_values, y_values[0], marker='o', label='Baseline (no fusion)')
  # ax.plot(x_values, y_values[2], marker='o', label='Baseline + fusion loss')
  # # ax.plot(x_values, y_values[1], marker='o', label='Only Fusion with EMA')
  # ax.plot(x_values, y_values[3], marker='o', label='2 stage training')
  # ax.plot(x_values, y_values[4], marker='o', label='2 stage training harder')
  # ax.plot(x_values, y_values[5], marker='o', label='Fusion')
  # ax.plot(x_values, y_values[6], marker='o', label='12 blocks')
  # ax.plot(x_values, y_values[7], marker='o', label='All2all')
  # ax.plot(x_values, y_values[8], marker='o', label='Text proj')
  # ax.plot(x_values, y_values[9], marker='o', label='Fusion updated')
  # ax.plot(x_values, y_values[10], marker='o', label='Fusion updated + EMA')
  # ax.plot(x_values, y_values[11], marker='o', label='Text encoder fine tuning')
  # ax.plot(x_values, y_values[12], marker='o', label='2 stage other ll')
  # ax.plot(x_values, y_values[13], marker='o', label='Multi positive')
  # ax.plot(x_values, y_values[14], marker='o', label='Multi positive no text')
  # ax.plot(x_values, y_values[15], marker='o', label='Multi positive text reconstruction')
  # ax.plot(x_values, y_values[16], marker='o', label='Multi positive sentence random')
  # ax.plot(x_values, y_values[17], marker='o', label='Multi positive sentence random maskcell')
  # ax.plot(x_values, y_values[18], marker='o', label='Multi positive sentence random maskcell text ft')
  ax.plot(x_values, y_values[19], marker='o', label='No fusion text ft')
  # ax.plot(x_values, y_values[20], marker='o', label='Fusion')
  # ax.plot(x_values, y_values[21], marker='o', label='Fusion text ft')
  # ax.plot(x_values, y_values[22], marker='o', label='Fusion no closer 200m')
  # ax.plot(x_values, y_values[23], marker='o', label='Fusion no closer 1000m')
  # # ax.plot(x_values, y_values[24], marker='o', label='Fusion no closer 1000m text ft')
  # ax.plot(x_values, y_values[25], marker='o', label='No fusion text frozen')
  # ax.plot(x_values, y_values[26], marker='o', label='No fusion multi-similarity')
  # ax.plot(x_values, y_values[27], marker='o', label='2 stage fusion')
  # ax.plot(x_values, y_values[28], marker='o', label='2 stage fusion modality encoding')
  # ax.plot(x_values, y_values[29], marker='o', label='2 stage fusion text reconstruction')
  # if i in [2, 3]:
  #   ax.plot(x_values, y_values[-3], marker='o', label='GeoClip')
  # if i==3:
  #   ax.plot(x_values, y_values[-2], marker='o', label='Multi positive text reconstruction - Level 18')
    # ax.plot([10, 100, 200, 500, 1000, 10000], [0.0223,0.5748,0.6035,0.6445,0.655,0.7262], marker='o', label='Lama cell codes Siglip')
  #   ax.plot([10, 100, 200, 500, 1000, 5000, 10000], y_values[-1], marker='o', label='With no text')
  ax.set_title(graph_title)
  ax.set_xlabel('Within X m')
  ax.set_ylabel('Accuracy')
  ax.set_xticks(x_values)
  ax.set_xscale('log')
  ax.legend()
  ax.grid(True)

plt.tight_layout()
plt.show()

## Cell codes per urbanization

In [ ]:
y_data = {
    'cell_codes all': [
        [0.0053,0.1024,0.264,0.3479,0.3997,0.4335,0.5417,0.6142],
        [0.0044,0.0862,0.2462,0.3446,0.413,0.4479,0.5492,0.6161],
        [0.0091, 0.0822, 0.237, 0.5673, 0.6257, 0.6372, 0.7125],
        [0.0046,0.0924,0.2626,0.3616,0.4219,0.454,0.5506,0.616],
        [0.0048,0.1012,0.285,0.3866,0.4435,0.4743,0.5664,0.6284],
    ],
    'cell_codes Rural': [
        [0.0041,0.0836,0.2445,0.3316,0.3695,0.3861,0.4229,0.4505],
        [0.0035,0.0689,0.2059,0.2933,0.3367,0.3527,0.3892,0.4229],
        [0.0088, 0.0779, 0.2294, 0.533, 0.5759, 0.5804, 0.603],
        [0.0045,0.0683,0.2172,0.3108,0.358,0.3721,0.4072,0.4397],
        [0.0035,0.0769,0.2368,0.3304,0.3711,0.3859,0.4165,0.4478],
    ],
    # 'cell_codes Sparse': [
    #     [0.006,0.105,0.3144,0.435,0.4817,0.4977,0.5304,0.562],
    #     [0.005,0.0889,0.2697,0.3968,0.4621,0.4751,0.5168,0.546],
    # ],
    'cell_codes Town or settlement': [
        [0.0041,0.0864,0.2344,0.308,0.3443,0.3629,0.4147,0.4593],
        [0.0029,0.0678,0.2022,0.2807,0.3256,0.3447,0.3957,0.443],
        [0.0079, 0.0765, 0.2238, 0.524, 0.566, 0.5714, 0.6052],
        [0.0031,0.0726,0.2176,0.2977,0.3396,0.3576,0.4053,0.4502],
        [0.0035,0.0805,0.2372,0.3208,0.3598,0.3769,0.4221,0.4629],
    ],
    'cell_codes Urban high density': [
        [0.0065,0.1252,0.308,0.4169,0.5234,0.6078,0.8157,0.8855],
        [0.0054,0.116,0.3138,0.4464,0.5786,0.661,0.8361,0.8966],
        [0.0095, 0.0815, 0.2376, 0.5721, 0.6566, 0.694, 0.8594],
        [0.0059,0.1156,0.3195,0.4471,0.5637,0.6405,0.8211,0.8838],
        [0.0065,0.1266,0.3449,0.4779,0.5872,0.6632,0.8322,0.889],
    ],
    'cell_codes Urban low density': [
        [0.0057,0.1021,0.263,0.3469,0.3969,0.4286,0.5327,0.6144],
        [0.0046,0.0857,0.2459,0.3461,0.4121,0.4446,0.5442,0.618],
        [0.0098, 0.0835, 0.2399, 0.5807, 0.6409, 0.6507, 0.7221],
        [0.005,0.0932,0.2637,0.3643,0.4229,0.4518,0.5446,0.6167],
        [0.0052,0.102,0.286,0.3895,0.4447,0.4732,0.5632,0.6332],
    ],
    'cell_codes Urban medium density': [
        [0.0059,0.1175,0.2894,0.3794,0.444,0.4922,0.6587,0.7539],
        [0.0055,0.1032,0.2851,0.3977,0.4876,0.5381,0.6909,0.7738],
        [0.0095,0.086,0.246,0.5941,0.6654,0.6829,0.8003],
        [0.0055,0.1109,0.3021,0.4144,0.4898,0.5372,0.6846,0.7669],
        [0.0057,0.1202,0.3278,0.4412,0.5136,0.5578,0.6976,0.7757],
    ],

}

x_values = [10, 50, 100, 200, 500, 1000, 5000, 10000]

fig, axes = plt.subplots(3, 2, figsize=(12, 10))
axes = axes.flatten()

for i, (graph_title, y_values) in enumerate(y_data.items()):
  graph_title = 'Retrieval with ' + graph_title + ' database'
  ax = axes[i]
  ax.plot(x_values, y_values[0], marker='o', label='Text reconstruction')
  # ax.plot(x_values, y_values[1], marker='o', label='Sentence random')
  ax.plot([10, 50, 100, 200, 500, 1000, 10000], y_values[2], marker='o', label='Lama Siglip')
  ax.plot(x_values, y_values[3], marker='o', label='Sentence random maskcell')
  ax.plot(x_values, y_values[4], marker='o', label='Sentence random maskcell text ft')
  ax.set_title(graph_title)
  ax.set_xlabel('Within X m')
  ax.set_ylabel('Accuracy')
  ax.set_xticks(x_values)
  ax.set_xscale('log')
  ax.legend()
  ax.grid(True)

plt.tight_layout()
plt.show()